In [11]:
import json
import numpy as np
# !pip install pandas matplotlib datasets peft
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset

from peft import LoraConfig, get_peft_model, TaskType
# !pip install scikit-learn
from sklearn.metrics import accuracy_score
from collections import defaultdict

from scipy.stats import norm
# !pip install sentence-transformers
# !pip install datasets rouge-score peft evaluate
# !pip install sentencepiece
# !pip install evaluate
# !pip install laplace-torch
# !pip install jaxtyping
# !pip install transformers
# !pip install sentence-transformers
import evaluate
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn as nn
import torch.optim as optim

import re

from tqdm import tqdm
import torch.nn.functional as F
from torch.func import functional_call

from torch.optim import AdamW
import random
from sklearn.model_selection import train_test_split
from copy import deepcopy
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer


from torch.nn import Module
from torch.utils.hooks import RemovableHandle

from peft import PeftModel
from copy import deepcopy
import copy
from collections import Counter

! pip install openai

from openai import OpenAI

import math
# !pip install gpytorch

import gpytorch
from gpytorch.likelihoods import _OneDimensionalLikelihood
from gpytorch.variational import VariationalStrategy, CholeskyVariationalDistribution
# !pip install scipy
from numpy.polynomial.hermite import hermgauss
from contextlib import contextmanager
import hashlib
# ! pip install hf-transfer


/Users/blackrose/miniconda3/envs/baylora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# print(torch.cuda.get_device_name(0))
# print(torch.cuda.is_available())
# print(torch.cuda.device_count())

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("Device:", device)

Device: cpu


In [13]:
N_WARMUP_BUCKETS = 256
WARMUP_BUCKET_TOKENS = [f"<W{i:03d}>" for i in range(N_WARMUP_BUCKETS)]

N_AL_BUCKETS = 4096
AL_BUCKET_TOKENS = [f"<U{i:04d}>" for i in range(N_AL_BUCKETS)]

UNK_USER_TOKEN = "<UUNK>"
ALL_USER_TOKENS = WARMUP_BUCKET_TOKENS + AL_BUCKET_TOKENS + [UNK_USER_TOKEN]


def hash_to_bucket(user_id: str, n_buckets: int) -> int:
    if user_id is None:
        return 0
    h = hashlib.md5(str(user_id).encode("utf-8")).hexdigest()
    return int(h, 16) % n_buckets

def normalize_user_token(raw_user_id: str, phase: str) -> str:
    if phase == "warmup":
        b = hash_to_bucket(raw_user_id, N_WARMUP_BUCKETS)
        return f"<W{b:03d}>"
    elif phase == "al":
        b = hash_to_bucket(raw_user_id, N_AL_BUCKETS)
        return f"<U{b:04d}>"
    else:
        return UNK_USER_TOKEN


seed = 10
def seed_everything(seed=seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(seed)


class GPT2PreferenceDataset(Dataset):
    def __init__(self, df, tokenizer, phase, max_length=512, max_prompt_len=256, max_resp_len=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.phase = phase
        self.max_length = max_length
        self.max_prompt_len = max_prompt_len
        self.max_resp_len = max_resp_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        raw_user_id = row.get("user_id", None)
        user_tok = normalize_user_token(raw_user_id, self.phase)
        
        post = row["post"]
        win = row["chosen_summary"]
        lose = row["rejected_summary"]

        # prompt = f"<UserID:{user_id}>\nPost:\n{post}\nAssistant:"
        prompt = f"{user_tok}\nPost:\n{post}\nAssistant:"

        # tokenize prompt separately (truncate prompt only)
        prompt_ids = self.tokenizer(prompt, add_special_tokens=False, truncation=True, max_length=self.max_prompt_len).input_ids

        # tokenize responses separately (truncate response only)
        win_ids_resp = self.tokenizer(win, add_special_tokens=False, truncation=True, max_length=self.max_resp_len).input_ids

        lose_ids_resp = self.tokenizer(lose, add_special_tokens=False, truncation=True, max_length=self.max_resp_len).input_ids

        # concat + final cap
        win_ids = (prompt_ids + win_ids_resp)[: self.max_length]
        lose_ids = (prompt_ids + lose_ids_resp)[: self.max_length]

        # prompt length AFTER truncation is just len(prompt_ids)
        # we need to no boundary of prompttoken in order to mask them off while cal. log-prob of response
        actual_prompt_len = len(prompt_ids)

        return {
            "preferred_ids": torch.tensor(win_ids, dtype=torch.long),
            "dispreferred_ids": torch.tensor(lose_ids, dtype=torch.long),
            "prompt_len": torch.tensor(actual_prompt_len, dtype=torch.long)}


# dynamic padding by collate_fn
# since the model expects a square tensor, we need to flatten the sequence length so that the dimensions match
def collate_fn(batch):
    pref_list = [b["preferred_ids"] for b in batch]
    disp_list = [b["dispreferred_ids"] for b in batch]
    prompt_lens = torch.stack([b["prompt_len"] for b in batch])

    pref_lens = torch.tensor([x.size(0) for x in pref_list], dtype=torch.long)
    disp_lens = torch.tensor([x.size(0) for x in disp_list], dtype=torch.long)

    max_len = max(pref_lens.max().item(), disp_lens.max().item())

    def pad_seq(seq_list, max_len, pad_id):
        return torch.stack([F.pad(x, (0, max_len - x.size(0)), value=pad_id) for x in seq_list])

    pad_id = tokenizer.pad_token_id
    preferred_ids   = pad_seq(pref_list, max_len, pad_id)
    dispreferred_ids = pad_seq(disp_list, max_len, pad_id)

    pos = torch.arange(max_len).unsqueeze(0)  # [1, L]
    preferred_mask   = (pos < pref_lens.unsqueeze(1)).long()
    dispreferred_mask = (pos < disp_lens.unsqueeze(1)).long()

    return {
        "preferred_ids": preferred_ids,
        "dispreferred_ids": dispreferred_ids,
        "preferred_mask": preferred_mask,
        "dispreferred_mask": dispreferred_mask,
        "prompt_lengths": prompt_lens}


# get_response_logprob (mask prompt + pad)
def get_response_logprob(logits, labels, prompt_lengths, tokenizer, attention_mask): # logits = output from model
    """
    cal log prob if model output (not include prompt)
    this will be used in DPO
    """

    # log softmax over vocab
    log_probs = F.log_softmax(logits, dim=-1) # [B, L, V]  log P(token|context up to i-1) log-probabilities of every vocabulary token at each position

    # shift logits and labels in order to make aligment btw them
    #  logits at position i predict the token at position i+1. Therefore, labels[i+1] is the token that logits[i] is predicting.
    log_probs = log_probs[:, :-1, :] # [B, L-1, V]
    labels_shifted = labels[:, 1:] # [B, L-1]

    # token_lps = torch.gather(log_probs, -1, labels.unsqueeze(-1)).squeeze(-1) # [B, L]
    token_lps = torch.gather(log_probs, -1, labels_shifted.unsqueeze(-1)).squeeze(-1) # (B, L-1) # token_lps[b, i] = log P( token_i | context ) // The log-probability of the true token

    B, L = labels_shifted.size() # NEW: L-1

    # make a position matrix [1..L] broadcasted
    positions = torch.arange(L, device=labels.device).unsqueeze(0) # [1, L-1] # prompt_lengths = no. of token prompt after truncate

    # mask summary region (no prompt)
    mask_summary = positions >= (prompt_lengths.unsqueeze(1) - 1) # [B, L-1]
    # mask_summary = positions >= prompt_lengths.unsqueeze(1)

    mask_att = attention_mask[:, 1:] == 1
    # padding mask (will not count log prob here)
    mask_notpad = (labels_shifted != tokenizer.pad_token_id)


    # final mask = summary region w/ not padding
    mask = (mask_summary & mask_att).float()

    # mask = (mask_summary & mask_att & mask_notpad).float()

    # compute log-likelihood (only summary tokens)
    resp_logprob = (token_lps * mask).sum(dim=1) # [B]

    return resp_logprob # sum logprob only response tokens





def dpo_loss_and_metrics(policy_logp_win, policy_logp_loss, ref_logp_win, ref_logp_loss, beta=0.1):
    """
    calculate DPO loss based on eq. 7 eq.7 of Rafailov et al.
    Loss DPO = - E_{(x, y_w, y_l) ~ D} [ log sigmoid(beta * ((log pseta(y_w|x) - log pseta(y_l|x)) - (log pref(y_w|x) - log pref(y_l|x))))]
    where:
        y_w = preferred (winner) response
        y_l = dispreferred (loser) response
        pseta  = policy model
        pref = reference model
        beta = temperature scaling factor
    """

    score_diff = beta * ((policy_logp_win - policy_logp_loss) - (ref_logp_win - ref_logp_loss))

    per_sample_loss = -F.logsigmoid(score_diff)
    loss = per_sample_loss.mean() # avg across batch

    with torch.no_grad():
        win_prob = torch.sigmoid(score_diff).mean().item()
        pairwise_accuracy = (policy_logp_win > policy_logp_loss).float().mean().item()
        margin_mean = (policy_logp_win - policy_logp_loss).mean().item() # learning progress

    return loss, {"win_prob": win_prob, # how confident is the model that it should choose the winning summary?
        "pairwise_accuracy": pairwise_accuracy, # does the model select the summary that wins? policy_logp_win > policy_logp_loss?
        "policy_margin": margin_mean}


# evaluation
def evaluate_dpo(policy, reference, dataloader, device, beta=0.1):
    policy.eval()
    reference.eval()
    total_pairwise = 0
    total_winprob = 0
    total_samples = 0

    with torch.no_grad():
        for batch in dataloader:
            logits_preferred = policy(input_ids=batch["preferred_ids"].to(device), attention_mask=batch["preferred_mask"].to(device)).logits # policy cal logit
            logits_dispreferred = policy(input_ids=batch["dispreferred_ids"].to(device), attention_mask=batch["dispreferred_mask"].to(device)).logits # policy cal logit

            # chnage logit to log prob (only summary)
            policy_logp_preferred = get_response_logprob(logits_preferred, batch["preferred_ids"].to(device), batch["prompt_lengths"].to(device), tokenizer,batch["preferred_mask"].to(device))
            policy_logp_dispreferred = get_response_logprob(logits_dispreferred, batch["dispreferred_ids"].to(device), batch["prompt_lengths"].to(device), tokenizer,batch["dispreferred_mask"].to(device))

            # cal logit
            logits_preferred_ref = reference(input_ids=batch["preferred_ids"].to(device), attention_mask=batch["preferred_mask"].to(device)).logits
            logits_dispreferred_ref = reference(input_ids=batch["dispreferred_ids"].to(device), attention_mask=batch["dispreferred_mask"].to(device)).logits

            # chnage logit to log prob (only summary)
            ref_logp_preferred = get_response_logprob(logits_preferred_ref, batch["preferred_ids"].to(device), batch["prompt_lengths"].to(device), tokenizer,batch["preferred_mask"].to(device))
            ref_logp_dispreferred = get_response_logprob(logits_dispreferred_ref, batch["dispreferred_ids"].to(device), batch["prompt_lengths"].to(device), tokenizer,batch["dispreferred_mask"].to(device))

            _, metrics = dpo_loss_and_metrics(policy_logp_preferred, policy_logp_dispreferred, ref_logp_preferred, ref_logp_dispreferred, beta=beta)

            total_pairwise += metrics['pairwise_accuracy'] * batch["preferred_ids"].size(0)
            total_winprob += metrics['win_prob'] * batch["preferred_ids"].size(0)
            total_samples += batch["preferred_ids"].size(0)

    avg_pairwise = total_pairwise / total_samples
    avg_winprob = total_winprob / total_samples
    return avg_pairwise, avg_winprob



# training loop
def train_dpo_gpt2(policy, reference, dataloader_train, dataloader_test, optimizer, device, epochs=5, beta=0.1):
    """" Training DPO using the loss to control parameter weight updates.
            Here, there is no separate reward model, as we assume the policy itself is optimised to replace the reward in the equation.
            This implementation is based on the DPO paper.
            The output of the training loop includes Pairwise Accuracy to show how often the model chooses the preferred summary,
            and DPO Win-Probability to indicate the model's confidence in choosing the winner over the loser
    """

    margin_history = []

    for epoch in tqdm(range(epochs), desc="Epochs", leave=True):
        policy.train()
        running_loss = 0.0

        for batch in dataloader_train:
            optimizer.zero_grad()

            logits_preferred = policy(input_ids=batch["preferred_ids"].to(device), attention_mask=batch["preferred_mask"].to(device)).logits
            logits_dispreferred = policy(input_ids=batch["dispreferred_ids"].to(device), attention_mask=batch["dispreferred_mask"].to(device)).logits

            policy_logp_preferred = get_response_logprob(logits_preferred, batch["preferred_ids"].to(device), batch["prompt_lengths"].to(device), tokenizer, batch["preferred_mask"].to(device)) # log p(preferred | prompt)
            policy_logp_dispreferred = get_response_logprob(logits_dispreferred, batch["dispreferred_ids"].to(device), batch["prompt_lengths"].to(device), tokenizer, batch["dispreferred_mask"].to(device))


            margin_batch = (policy_logp_preferred - policy_logp_dispreferred).detach().cpu().numpy()
            avg_margin = margin_batch.mean()
            # append to a list or print
            margin_history.append(avg_margin) # if model perform well, margin will increase

            # print("policy_logp (sample):", policy_logp_preferred[0].item())
            # print("logp diff (win - lose):", (policy_logp_preferred - policy_logp_dispreferred)[0].item())

            with torch.no_grad(): # not train in ref model
                logits_preferred_ref = reference(input_ids=batch["preferred_ids"].to(device), attention_mask=batch["preferred_mask"].to(device)).logits
                logits_dispreferred_ref = reference(input_ids=batch["dispreferred_ids"].to(device), attention_mask=batch["dispreferred_mask"].to(device)).logits

                ref_logp_preferred = get_response_logprob(logits_preferred_ref, batch["preferred_ids"].to(device), batch["prompt_lengths"].to(device), tokenizer,batch["preferred_mask"].to(device))
                ref_logp_dispreferred = get_response_logprob(logits_dispreferred_ref, batch["dispreferred_ids"].to(device), batch["prompt_lengths"].to(device), tokenizer, batch["dispreferred_mask"].to(device))

            loss, _ = dpo_loss_and_metrics(policy_logp_preferred, policy_logp_dispreferred, ref_logp_preferred, ref_logp_dispreferred, beta=beta)
            loss.backward()

            # debug check gradient magnitude
            # for name, p in policy.named_parameters():
            #     if p.grad is not None:
            #         print("grad", name, p.grad.abs().mean().item())

            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(dataloader_train)
        train_acc, train_win = evaluate_dpo(policy, reference, dataloader_train, device, beta)
        test_acc, test_win = evaluate_dpo(policy, reference, dataloader_test, device, beta)

        print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Train Accuracy: {train_acc:.4f}, Win Probability: {train_win:.4f} | Validation Accuracy: {test_acc:.4f}, Win Probability: {test_win:.4f}")
    return margin_history

# load data
with open("batch14.json", "r", encoding="utf-8") as f:
    lines = f.readlines()
data = [json.loads(line) for line in lines]

rows = []
for example in data:
    post = example.get("info", {}).get("post", "")
    choice = example.get("choice", None)
    summaries = example.get("summaries", [])
    extra = example.get("extra", {})
    user_id = example.get("worker", None)

    confidence = extra.get("confidence", None) if isinstance(extra, dict) else None

    if len(summaries) >= 2:
        summary_0 = summaries[0].get("text", "")
        summary_1 = summaries[1].get("text", "")
    else:
        summary_0 = ""
        summary_1 = ""

    rows.append({"user_id": user_id,
                "post": post,
                "summary_0": summary_0,
                "summary_1": summary_1,
                "choice": choice,
                "confidence": confidence})



df = pd.DataFrame(rows)
df['chosen_summary'] = df.apply(lambda row: row['summary_0'] if row['choice']==0 else row['summary_1'], axis=1)
df['rejected_summary'] = df.apply(lambda row: row['summary_1'] if row['choice']==0 else row['summary_0'], axis=1)
df = df.dropna(subset=["post","chosen_summary"]).reset_index(drop=True)
print(f"before cut {len(df)}")
df = df[df['post'].str.len().between(0, 1000)].reset_index(drop=True)
# df = df[:50]
print(f"after cut {len(df)}")

# split train/test
train_df, test_df = train_test_split(df, test_size=0.1, random_state=10)


before cut 4008
after cut 1277


In [14]:
data

[{'info': {'id': 't3_3sm16t',
   'post': "Kind of an unconventional post here, but here it goes:\n\nMy roommate is a guy that smokes a ton of weed at just about any hour of the day. In fact, I heard him fill up his bong at 9 this morning in the sink near my room. I'm cool with him smoking weed, I mean you do you, bud. What I'm not cool with is his blasting music when I'm trying to do work (we share a wall), the awful smell that comes from his room seeping in to mine (again, right next to each other), and the sheer number of people that come over for an indefinite amount of time (the door is literally unlocked so his friends can walk in whenever). These guys use our water at a rate that I'm not okay to split with my roommie at our agreed 50-50 rate. These guys use our toilet paper at a rate that I can't afford and have been stuck in a situation where I've had to hobble out to the drug store to buy more twice so far. These guys order pizza every night and then my roommate makes a big dea

In [15]:
df

,user_id,post,summary_0,summary_1,choice,confidence,chosen_summary,rejected_summary
0,OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca,The funding behind a lot of thinks is pretty s...,Why are we so fascinated by space/mars in gen...,Why do we fund space research/expeditions/wha...,1,6,Why do we fund space research/expeditions/wha...,Why are we so fascinated by space/mars in gen...
1,OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca,The funding behind a lot of thinks is pretty s...,"Why are we so interested in space, specifical...",Why are we so fascinated by space/mars in gen...,1,7,Why are we so fascinated by space/mars in gen...,"Why are we so interested in space, specifical..."
2,OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca,The funding behind a lot of thinks is pretty s...,"Why are we so interested in space, specifical...",Why do we fund space research/expeditions/wha...,1,6,Why do we fund space research/expeditions/wha...,"Why are we so interested in space, specifical..."
3,OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca,The funding behind a lot of thinks is pretty s...,"Why are we so interested in space, specifical...",Why are we so obsessed with space in general ...,1,7,Why are we so obsessed with space in general ...,"Why are we so interested in space, specifical..."
4,OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca,The funding behind a lot of thinks is pretty s...,Why are we so obsessed with space in general ...,Why do we fund space research/expeditions/wha...,1,7,Why do we fund space research/expeditions/wha...,Why are we so obsessed with space in general ...
...,...,...,...,...,...,...,...,...
1272,P2p07Up4eJyvxrrVYgwtb60krbFbxI,"Hi /r/relationships, this may sound confusing ...","In a relationship with GF, GF slept with best...",In a relationship with G but think I'm in lov...,0,2,"In a relationship with GF, GF slept with best...",In a relationship with G but think I'm in lov...
1273,P2p07Up4eJyvxrrVYgwtb60krbFbxI,"Hi /r/relationships, this may sound confusing ...","In a relationship with GF, GF slept with best...",in a relationship with g but think I'm in lov...,0,3,"In a relationship with GF, GF slept with best...",in a relationship with g but think I'm in lov...
1274,P2p07Up4eJyvxrrVYgwtb60krbFbxI,"Hi /r/relationships, this may sound confusing ...","In a relationship with GF, recently GF slept ...",In a relationship with G but think I'm in lov...,0,2,"In a relationship with GF, recently GF slept ...",In a relationship with G but think I'm in lov...
1275,P2p07Up4eJyvxrrVYgwtb60krbFbxI,"Hi /r/relationships, this may sound confusing ...",in a relationship with g but think I'm in lov...,In a relationship with G but think I'm in lov...,1,6,In a relationship with G but think I'm in lov...,in a relationship with g but think I'm in lov...


In [ ]:
# model n LoRA
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.truncation_side = "right"

base = AutoModelForCausalLM.from_pretrained(model_name).to(device)
base.config.use_cache = False

special = {"additional_special_tokens": ALL_USER_TOKENS}
num_added = tokenizer.add_special_tokens(special)
if num_added > 0:
    base.resize_token_embeddings(len(tokenizer))
    
reference = copy.deepcopy(base).to(device)
policy_base = copy.deepcopy(base).to(device)

# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn", "c_proj"], #"c_fc" # GPT-2 attention qkv // qwen "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM")

policy = get_peft_model(policy_base, lora_config).to(device)
policy.config.use_cache = False


# freeze reference (after all resizing and cloning)
reference.eval()
reference.config.use_cache = False
for p in reference.parameters():
    p.requires_grad = False


# freeze non-LoRA params on policy
for name, param in policy.named_parameters():
    if "lora_" in name.lower():
        param.requires_grad = True
    else:
        param.requires_grad = False


policy.print_trainable_parameters()

# using AdamW optimizer and update only LoRA parameter
optimizer = AdamW(filter(lambda p: p.requires_grad, policy.parameters()), lr=5e-5)

with torch.no_grad():
    w_pol = policy.get_input_embeddings().weight
    w_ref = reference.get_input_embeddings().weight
    print("max abs diff emb:", (w_pol - w_ref).abs().max().item())

trainable_ref = [n for n,p in reference.named_parameters() if p.requires_grad]
print("trainable reference params:", trainable_ref[:10], "count=", len(trainable_ref))

# prepare data
train_dataset = GPT2PreferenceDataset(train_df, tokenizer, phase="warmup", max_length=512, max_prompt_len=256, max_resp_len=256)
test_dataset  = GPT2PreferenceDataset(test_df,  tokenizer, phase="warmup", max_length=512, max_prompt_len=256, max_resp_len=256)

dataloader_train = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
dataloader_test = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

# train
margin_history = train_dpo_gpt2(policy, reference, dataloader_train, dataloader_test, optimizer, device, epochs=3, beta=0.2)

# # save LoRA
policy.save_pretrained("gpt2-dpo-lora")
tokenizer.save_pretrained("gpt2-dpo-lora")

with torch.no_grad():
    print("max abs diff lm_head:", (policy.base_model.model.lm_head.weight - reference.lm_head.weight).abs().max().item())

print("After this, should not print anything")
for n, p in policy.named_parameters():
    if p.requires_grad and ("lora_" not in n.lower()):
        print("non-LoRA trainable:", n, p.shape)



The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 811,008 || all params: 128,593,920 || trainable%: 0.6307
max abs diff emb: 0.0
trainable reference params: [] count= 0


Epochs:  33%|███▎      | 1/3 [01:29<02:58, 89.27s/it]

Epoch 1 | Loss: 0.8250 | Train Accuracy: 0.5413, Win Probability: 0.5506 | Validation Accuracy: 0.5781, Win Probability: 0.5209


Epochs:  67%|██████▋   | 2/3 [02:13<01:02, 62.99s/it]

Epoch 2 | Loss: 0.7279 | Train Accuracy: 0.5631, Win Probability: 0.6158 | Validation Accuracy: 0.6016, Win Probability: 0.5611


Epochs: 100%|██████████| 3/3 [02:49<00:00, 56.50s/it]

Epoch 3 | Loss: 0.6603 | Train Accuracy: 0.5718, Win Probability: 0.6534 | Validation Accuracy: 0.6094, Win Probability: 0.5920



/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


max abs diff lm_head: 0.0
After this, should not print anything


In [ ]:
def batch_outer_sum_masked_full(x: torch.Tensor, mask: torch.Tensor, *, out_dtype=torch.float32):
    """
    Full Fisher: returns [D,D]
    cal cov. full matrix
    """
    if x.dim() == 3:
        B, T, D = x.shape
        m = mask[:, :T].to(dtype=x.dtype)
        vec = (x * m.unsqueeze(-1)).reshape(-1, D) # [N,D]
    elif x.dim() == 2:
        N, D = x.shape
        m = mask[:N].to(dtype=x.dtype)
        vec = x * m.unsqueeze(-1) # [N,D]
    else:
        raise ValueError("x must be 2D or 3D")

    vec32 = vec.to(torch.float32)
    return (vec32.transpose(0, 1) @ vec32).to(out_dtype) # [D,D]


def batch_outer_sum_masked_diag(x: torch.Tensor, mask: torch.Tensor, *, out_dtype=torch.float32):
    """
    Diag: returns [D]
    Only diagonal Fisher
    """
    if x.dim() == 3:
        B, T, D = x.shape
        m = mask[:, :T].to(dtype=x.dtype)
        vec = (x * m.unsqueeze(-1)).reshape(-1, D) # [N,D]
    elif x.dim() == 2:
        N, D = x.shape
        m = mask[:N].to(dtype=x.dtype)
        vec = x * m.unsqueeze(-1) # [N,D]
    else:
        raise ValueError("x must be 2D or 3D")

    vec32 = vec.to(torch.float32)
    return (vec32 * vec32).sum(dim=0).to(out_dtype) # [D]


KFAC_MODE = "diag"   # or "full"

if KFAC_MODE == "full":
    batch_outer_sum_masked = batch_outer_sum_masked_full
    k_rank = 4
    K_dirs = 8
    M_resid = 3
else:
    batch_outer_sum_masked = batch_outer_sum_masked_diag
    k_rank = 0
    K_dirs = 0
    M_resid = 20


class KFACContext:
    def __init__(self):
        self.mask = None
        self.masks = None
        self.forward_idx = 0
        self.mask_stack = defaultdict(list)


def register_root_mask_switch_hooks(model: nn.Module, ctx: KFACContext):
    def pre_hook(module, inputs):
        masks = getattr(ctx, "masks", None)
        if not masks:
            ctx.mask = None
            return
        fi = getattr(ctx, "forward_idx", 0)
        ctx.mask = masks[min(fi, len(masks) - 1)]

    def post_hook(module, inputs, outputs):
        ctx.forward_idx = getattr(ctx, "forward_idx", 0) + 1

    h1 = model.register_forward_pre_hook(pre_hook)
    h2 = model.register_forward_hook(post_hook)
    return [h1, h2]


def _accum_inplace(buf: dict, key: str, mat: torch.Tensor):
    if key not in buf:
        buf[key] = mat.clone()
    else:
        buf[key].add_(mat)


def save_input_hook(layer_name: str, activations_dict: dict, ctx: KFACContext,
                    *, offload_to_cpu: bool = False, out_dtype=torch.float32):
    """A factor activations"""
    def hook(module, inputs):
        if not inputs or ctx.mask is None:
            return

        x = inputs[0].detach()

        # align mask m
        if x.dim() == 3:
            B, T, D = x.shape
            m = ctx.mask[:, :T].to(dtype=x.dtype)
            if m.shape[1] < T:
                m = F.pad(m, (0, T - m.shape[1]), value=0.0)
        elif x.dim() == 2:
            N, D = x.shape
            if ctx.mask.dim() == 2:
                m = ctx.mask.reshape(-1).to(dtype=x.dtype)
            else:
                m = ctx.mask.to(dtype=x.dtype)
            if m.numel() < N:
                m = F.pad(m, (0, N - m.numel()), value=0.0)
            else:
                m = m[:N]
        else:
            return

        ctx.mask_stack[layer_name].append(m)

        mat = batch_outer_sum_masked(x, m, out_dtype=out_dtype) # [D,D] if full else [D]
        if offload_to_cpu:
            mat = mat.cpu()
        _accum_inplace(activations_dict, layer_name, mat)

    return hook


def save_output_grad_hook(layer_name: str, grads_dict: dict, ctx: KFACContext,
                          *, offload_to_cpu: bool = False, out_dtype=torch.float32):
    """S factor  gradient covariance"""
    def hook(module, grad_input, grad_output):
        if not grad_output:
            return
        if len(ctx.mask_stack[layer_name]) == 0:
            return

        g = grad_output[0].detach()
        m = ctx.mask_stack[layer_name].pop()

        if g.dim() == 3:
            B, T, D = g.shape
            if m.dim() != 2:
                return
            m2 = m[:, :T].to(dtype=g.dtype)
            if m2.shape[1] < T:
                m2 = F.pad(m2, (0, T - m2.shape[1]), value=0.0)
            mat = batch_outer_sum_masked(g, m2, out_dtype=out_dtype)
        elif g.dim() == 2:
            N, D = g.shape
            if m.dim() == 2:
                m1 = m.reshape(-1).to(dtype=g.dtype)
            else:
                m1 = m.to(dtype=g.dtype)
            if m1.numel() < N:
                m1 = F.pad(m1, (0, N - m1.numel()), value=0.0)
            else:
                m1 = m1[:N]
            mat = batch_outer_sum_masked(g, m1, out_dtype=out_dtype)
        else:
            return

        if offload_to_cpu:
            mat = mat.cpu()
        _accum_inplace(grads_dict, layer_name, mat)

    return hook

def is_lora_module(name: str, module: nn.Module) -> bool:
    """K-FAC work only LoRA part not base model"""
    lname = name.lower()
    if ("lora_a" in lname) or ("lora_b" in lname) or (".lora_a" in lname) or (".lora_b" in lname):
        return hasattr(module, "weight") and (module.weight is not None)
    return False
    
def register_kfac_hooks(model: nn.Module, activations: dict, grads: dict, ctx: KFACContext,
                        *, offload_to_cpu: bool = False, out_dtype=torch.float32):
    hooks = []
    for name, module in model.named_modules():
        if is_lora_module(name, module):
            fwd = module.register_forward_pre_hook(
                save_input_hook(name, activations, ctx, offload_to_cpu=offload_to_cpu, out_dtype=out_dtype)
            )
            try:
                bwd = module.register_full_backward_hook(
                    save_output_grad_hook(name, grads, ctx, offload_to_cpu=offload_to_cpu, out_dtype=out_dtype),
                    prepend=False
                )
            except Exception:
                bwd = module.register_backward_hook(
                    save_output_grad_hook(name, grads, ctx, offload_to_cpu=offload_to_cpu, out_dtype=out_dtype)
                )
            hooks.extend([fwd, bwd])
    return hooks


def remove_hooks(hooks):
    for h in hooks:
        try: h.remove()
        except Exception: pass


def build_summary_token_mask(input_ids: torch.Tensor, attention_mask: torch.Tensor, prompt_len: torch.Tensor):
    """
    consistency with get_response_logprob:
      - labels_shifted = input_ids[:, 1:]
      - use same attention_mask[:, 1:]
      - response started positions >= prompt_len-1 (labels_shifted)
    return mask_full [B, L]
    """
    device = input_ids.device
    B, L = input_ids.shape

    pos = torch.arange(L - 1, device=device).unsqueeze(0) # [1, L-1]
    mask_summary = pos >= (prompt_len.unsqueeze(1) - 1) # [B, L-1]
    mask_att = attention_mask[:, 1:] == 1 # [B, L-1]

    mask = (mask_summary & mask_att).float() # [B, L-1]
    mask_full = F.pad(mask, (0, 1), value=0.0) # [B, L]
    return mask_full


def calculate_kfac_factors_dpo(policy, reference, loader, device, tokenizer, beta: float = 0.1, curvature_mode: str = "eval", max_steps: int = 50, offload_to_cpu: bool = False, out_dtype=torch.float32):
    """Compute (A,S) of the LoRA parameters under the DPO loss with a summary-token mask."""
    if curvature_mode == "train":
        policy.train()
    else:
        policy.eval()
    reference.eval()

    activations, grads = {}, {}
    ctx = KFACContext()

    hooks = register_kfac_hooks(policy, activations, grads, ctx, offload_to_cpu=offload_to_cpu, out_dtype=out_dtype)
    root_hooks = register_root_mask_switch_hooks(policy, ctx)

    total_token_mass = 0.0

    total = min(len(loader), max_steps) if max_steps is not None else len(loader)
    for step, batch in enumerate(tqdm(loader, total=total, desc="Collecting KFAC", leave=False)):
        if max_steps is not None and step >= max_steps:
            break
        
        win_ids   = batch["preferred_ids"].to(device)
        lose_ids  = batch["dispreferred_ids"].to(device)
        win_att   = batch["preferred_mask"].to(device)
        lose_att  = batch["dispreferred_mask"].to(device)
        prompt_ln = batch["prompt_lengths"].to(device)

        mask_win  = build_summary_token_mask(win_ids,  win_att,  prompt_ln)
        mask_lose = build_summary_token_mask(lose_ids, lose_att, prompt_ln)

        ctx.masks = [mask_win, mask_lose]
        ctx.forward_idx = 0
        ctx.mask_stack.clear()

        total_token_mass += float(mask_win.sum().item() + mask_lose.sum().item())

        policy.zero_grad(set_to_none=True)

        logits_win  = policy(input_ids=win_ids,  attention_mask=win_att,  use_cache=False).logits
        logits_lose = policy(input_ids=lose_ids, attention_mask=lose_att, use_cache=False).logits

        pol_logp_win  = get_response_logprob(logits_win,  win_ids,  prompt_ln, tokenizer, win_att)
        pol_logp_lose = get_response_logprob(logits_lose, lose_ids, prompt_ln, tokenizer, lose_att)

        with torch.no_grad():
            ref_logits_win  = reference(input_ids=win_ids,  attention_mask=win_att,  use_cache=False).logits
            ref_logits_lose = reference(input_ids=lose_ids, attention_mask=lose_att, use_cache=False).logits
            ref_logp_win  = get_response_logprob(ref_logits_win,  win_ids,  prompt_ln, tokenizer, win_att)
            ref_logp_lose = get_response_logprob(ref_logits_lose, lose_ids, prompt_ln, tokenizer, lose_att)

        loss, _ = dpo_loss_and_metrics(pol_logp_win, pol_logp_lose, ref_logp_win, ref_logp_lose, beta=beta)
        loss.backward()

    remove_hooks(hooks)
    remove_hooks(root_hooks)

    denom = max(total_token_mass, 1.0)
    factors = {}
    for name in activations.keys():
        A = activations[name] / denom
        S = grads.get(name, None)
        if S is None:
            continue
        S = S / denom
        factors[name] = (A, S)

    return factors


In [ ]:
# K-FAC cal curvature of the loss around the current parameter point, using reference outputs (correct summary in this case)' to compute the gradient covariance
factors = calculate_kfac_factors_dpo(
    policy=policy, reference=reference,
    loader=dataloader_train, # curvature on dataloader_train
    device=device, tokenizer=tokenizer, beta=0.2)

print("K-FAC factors:", factors.keys())


K-FAC factors: dict_keys(['base_model.model.transformer.h.0.attn.c_attn.lora_A.default', 'base_model.model.transformer.h.0.attn.c_attn.lora_B.default', 'base_model.model.transformer.h.0.attn.c_proj.lora_A.default', 'base_model.model.transformer.h.0.attn.c_proj.lora_B.default', 'base_model.model.transformer.h.0.mlp.c_proj.lora_A.default', 'base_model.model.transformer.h.0.mlp.c_proj.lora_B.default', 'base_model.model.transformer.h.1.attn.c_attn.lora_A.default', 'base_model.model.transformer.h.1.attn.c_attn.lora_B.default', 'base_model.model.transformer.h.1.attn.c_proj.lora_A.default', 'base_model.model.transformer.h.1.attn.c_proj.lora_B.default', 'base_model.model.transformer.h.1.mlp.c_proj.lora_A.default', 'base_model.model.transformer.h.1.mlp.c_proj.lora_B.default', 'base_model.model.transformer.h.2.attn.c_attn.lora_A.default', 'base_model.model.transformer.h.2.attn.c_attn.lora_B.default', 'base_model.model.transformer.h.2.attn.c_proj.lora_A.default', 'base_model.model.transformer.h.2

In [9]:
name = next(iter(factors.keys()))
A, S = factors[name]
print("layer:", name, "A:", A.shape, "S:", S.shape)

def offdiag_ratio(M):
    M = M.float()
    diag = torch.diag(M).abs().mean()
    off  = (M - torch.diag(torch.diag(M))).abs().mean()
    return (off / (diag + 1e-12)).item()

print("A off/diag:", offdiag_ratio(A))
print("S off/diag:", offdiag_ratio(S))
print("A symmetric err:", (A - A.T).abs().max().item())
print("S symmetric err:", (S - S.T).abs().max().item())


layer: base_model.model.transformer.h.0.attn.c_attn.lora_A.default A: torch.Size([768]) S: torch.Size([8])
A off/diag: 0.0
S off/diag: 0.0
A symmetric err: 0.0
S symmetric err: 0.0


/tmp/ipykernel_851/2202090784.py:13: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4421.)
  print("A symmetric err:", (A - A.T).abs().max().item())


In [10]:
for name, module in policy.named_modules():
    if hasattr(module, "lora_A"):
        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight
        print(f"[{name}] A.shape={A.shape}, B.shape={B.shape}")


[base_model.model.transformer.h.0.attn.c_attn] A.shape=torch.Size([8, 768]), B.shape=torch.Size([2304, 8])
[base_model.model.transformer.h.0.attn.c_proj] A.shape=torch.Size([8, 768]), B.shape=torch.Size([768, 8])
[base_model.model.transformer.h.0.mlp.c_proj] A.shape=torch.Size([8, 3072]), B.shape=torch.Size([768, 8])
[base_model.model.transformer.h.1.attn.c_attn] A.shape=torch.Size([8, 768]), B.shape=torch.Size([2304, 8])
[base_model.model.transformer.h.1.attn.c_proj] A.shape=torch.Size([8, 768]), B.shape=torch.Size([768, 8])
[base_model.model.transformer.h.1.mlp.c_proj] A.shape=torch.Size([8, 3072]), B.shape=torch.Size([768, 8])
[base_model.model.transformer.h.2.attn.c_attn] A.shape=torch.Size([8, 768]), B.shape=torch.Size([2304, 8])
[base_model.model.transformer.h.2.attn.c_proj] A.shape=torch.Size([8, 768]), B.shape=torch.Size([768, 8])
[base_model.model.transformer.h.2.mlp.c_proj] A.shape=torch.Size([8, 3072]), B.shape=torch.Size([768, 8])
[base_model.model.transformer.h.3.attn.c_at

In [ ]:
def param_to_module_key(param_name: str) -> str:
    """
    Map parameter name -> module name (for KFAC factors). Keeps lora_A / lora_B separation, removes only '.weight' / '.bias'
    As K-FAC factors are stored at the module level, while the Laplace variance is at the parameter level. Therefore, a central key is needed to correctly map between them
    """
    if param_name.endswith(".weight"):
        return param_name[:-len(".weight")]
    if param_name.endswith(".bias"):
        return param_name[:-len(".bias")]
    return param_name


def get_lora_param_dict(model):
    """
    Return trainable LoRA parameters only.
    """
    out = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        lname = name.lower()
        if ("lora_a" in lname or "lora_b" in lname) and (name.endswith(".weight") or name.endswith(".bias")):
            out[name] = p
    return out



def kron_diag(diag_A: torch.Tensor, diag_S: torch.Tensor, damping=1e-5, normalize=True):
    """Laplace Approximation requires the diagonal of the Fisher.
    while compute_kfac_factors gives us only the Kronecker factors A and S.
    kron_diag extracts diag(A) and diag(S). Therefore, this function must construct diag(A x S) (the Fisher diagonal)
    """
    # clamp
    A = diag_A.float().clamp_min(1e-8)
    S = diag_S.float().clamp_min(1e-8)

    # normalize
    if normalize:
        A = A / (A.norm() + 1e-12)
        S = S / (S.norm() + 1e-12)

    # damping
    A_damped = A + damping
    S_damped = S + damping

    # kronecker diag
    fisher_diag = torch.outer(S_damped, A_damped)

    # clamp final values
    fisher_diag = fisher_diag.clamp_min(1e-8)

    return fisher_diag




########################################################################################################################
@contextmanager
def lora_param_swap_inplace(model, new_params: dict):
    if not new_params:
        yield
        return
    backup = {}
    with torch.no_grad():
        for n, p in model.named_parameters():
            if n in new_params:
                backup[n] = p.data.clone()
                p.data.copy_(new_params[n].to(p.device, dtype=p.dtype))
    try:
        yield
    finally:
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in backup:
                    p.data.copy_(backup[n])



def build_dpo_batch_inputs(posts, summaries, tokenizer, device, raw_user_id, phase, max_prompt_len=256, max_resp_len=256, max_length=512):
    """construct input_ids, attention_mask, and prompt_lengths so that they exactly match the DPO definition, in order to compute the utility and log-probability only over the response part"""
    user_tok = normalize_user_token(raw_user_id, phase)
    prompts = [f"{user_tok}\nPost:\n{p}\nAssistant:" for p in posts]
    
    prompt_ids_list = tokenizer(
        prompts, add_special_tokens=False,
        truncation=True, max_length=max_prompt_len
    ).input_ids

    resp_ids_list = tokenizer(
        summaries, add_special_tokens=False,
        truncation=True, max_length=max_resp_len
    ).input_ids

    seqs = []
    prompt_lens = []
    seq_lens = []

    for pr_ids, rp_ids in zip(prompt_ids_list, resp_ids_list):
        ids = (pr_ids + rp_ids)[:max_length]
        seqs.append(torch.tensor(ids, dtype=torch.long))
        prompt_lens.append(len(pr_ids))
        seq_lens.append(len(ids))

    max_len = max(seq_lens)
    pad_id = tokenizer.pad_token_id

    input_ids = torch.stack([F.pad(x, (0, max_len - x.size(0)), value=pad_id) for x in seqs]).to(device)

    pos = torch.arange(max_len, device=device).unsqueeze(0)
    attention_mask = (pos < torch.tensor(seq_lens, device=device).unsqueeze(1)).long()

    prompt_lens = torch.tensor(prompt_lens, dtype=torch.long, device=device)
    return input_ids, attention_mask, prompt_lens


@torch.no_grad()
def score_dpo_per_example_from_params(policy, reference, params, tokenizer, posts, summaries, device,
    *, user_id, phase="al", max_length=512, batch_size=8, mode="sum", beta=1.0, max_prompt_len=256, max_resp_len=256):
    """
    Compute the DPO-style utility of each (post,summary) under a given set of LoRA parameters (e.g. a sample from the Laplace posterior)
    Utility score per example is u(x,y) = beta * (logp_policy(y|x) - logp_ref(y|x))
    where logp is computed on response tokens only (prompt-masked) and padding-masked.
    """
    
    policy.eval(); reference.eval()
    out = []

    # collect parameter sample (params)
    with lora_param_swap_inplace(policy, params): # input the params into model
        for i in range(0, len(posts), batch_size):
            bp = posts[i:i+batch_size]
            bs = summaries[i:i+batch_size]

            input_ids, attn, plens = build_dpo_batch_inputs(
                posts=bp,
                summaries=bs,
                tokenizer=tokenizer,
                device=device,
                raw_user_id=user_id, phase="al", 
                max_prompt_len=max_prompt_len,
                max_resp_len=max_resp_len,
                max_length=max_length)
            
            
            # cal log prob only in response
            # forward via model
            lp_pol = get_response_logprob(logits=policy(input_ids=input_ids, attention_mask=attn, use_cache=False).logits, labels=input_ids, prompt_lengths=plens, tokenizer=tokenizer, attention_mask=attn)
            lp_ref = get_response_logprob(logits=reference(input_ids=input_ids, attention_mask=attn, use_cache=False).logits, labels=input_ids,prompt_lengths=plens, tokenizer=tokenizer, attention_mask=attn)

            # utility of DPO
            score = beta * (lp_pol - lp_ref) # this eq comes from DPO; if theta is x, uti=?

            if mode == "avg":
                labels_shifted = input_ids[:, 1:]
                pos = torch.arange(labels_shifted.size(1), device=device).unsqueeze(0)
                mask = (pos >= (plens.unsqueeze(1) - 1)) & (attn[:, 1:] == 1)
                denom = mask.float().sum(dim=1).clamp_min(1.0)
                score = score / denom


            out.extend(score.detach().cpu().tolist())

    return np.asarray(out, dtype=np.float32)




In [ ]:
def stable_eigh_topk(M, k=2, eps=1e-6):
    """The LoRA parameters are extremely high-dimensional, so a full Laplace posterior would be enormous. 
    In practice, however, uncertainty is not spread evenly in all directions, only a few directions are truly important (the principal curvature directions). 
    This is why we construct rank-K directions. This function finds the most important eigenvectors (top-k) to extract the principal directions of the curvature (Fisher/Hessian)"""
    M = 0.5 * (M + M.T)
    M = M + eps * torch.eye(M.size(0), device=M.device, dtype=M.dtype)
    evals, evecs = torch.linalg.eigh(M)
    idx = torch.argsort(evals, descending=True)[:k]
    return evals[idx], evecs[:, idx]

def orthonormalize_dirs_frobenius(dirs, eps=1e-12):
    """
    We orthonormalize the rank-K LoRA direction matrices under the Frobenius inner product (used instead of the standard dot product in parameter-matrix space)
    to avoid redundancy. Without this step, multiple directions may point in nearly the same subspace, leading to double-counted uncertainty and artificially 
    inflated variance along certain directions. Orthonormalization also ensures a valid low-rank Gaussian posterior approximation, 
    so that the low-rank covariance can be written in the form: U @ Lambda @ U.T, which corresponds to a Gaussian distribution.
    """
    Q = []
    for u in dirs:
        v = u.clone()
        for q in Q:
            v = v - (v * q).sum() * q  # Frobenius inner product
        n = v.norm()
        if float(n) > eps:
            Q.append(v / (n + eps))
    return Q



def build_laplace_posterior_diag_and_rank4(
    policy,
    factors,
    prior_var: float = 0.01,
    k: int = k_rank,
    rank_eigs_A: int = 2,
    rank_eigs_S: int = 2,
    use_resid_subtraction: bool = True,
    eps: float = 1e-12):
    """
    This function constructs a Laplace posterior approximation over the LoRA parameters by decomposing the covariance into two parts:

    1) A diagonal component (covering all parameters), which captures per-parameter uncertainty.
    2) A low-rank component, consisting of the top-K most important directions (rank-K), which capture the principal curvature directions of the loss.
    
    The resulting posterior can be written as: p(theta) ≈ N(theta_hat, D_full [diagonal covariance] + sum_{i=1}^K sigma_i^2 * u_i @ u_i.T [rank-K low-rank covariance])

    where:
    - theta_hat is the MAP estimate of the LoRA parameters,
    - D_full is the full diagonal variance,
    - u_i are the orthonormal rank-K directions,
    - sigma_i^2 are the variances along those directions.

    This structure yields a scalable and expressive approximation to the full Laplace posterior

    This function supports two modes depending on factors[module_key] shapes:
      - FULL KFAC: A:[Din,Din], S:[Dout,Dout] -> posterior diag + rank-k directions from top eigvecs of A and S
      - DIAG KFAC: A:[Din], S:[Dout] -> posterior diag only (rank_dirs/rank_lambdas empty)
    """

    def stable_eigh_topk(M, k=2, eps_local=1e-6):
        """same as above"""
        M = 0.5 * (M + M.T)
        M = M + eps_local * torch.eye(M.size(0), device=M.device, dtype=M.dtype)
        evals, evecs = torch.linalg.eigh(M)
        idx = torch.argsort(evals, descending=True)[:k]
        return evals[idx], evecs[:, idx]

    def orthonormalize_dirs_frobenius(dirs, eps_local=1e-12):
        """same as above"""
        Q = []
        for u in dirs:
            v = u.clone()
            for q in Q:
                v = v - (v * q).sum() * q  # Frobenius inner product
            n = v.norm()
            if float(n) > eps_local:
                Q.append(v / (n + eps_local))
        return Q

    def param_to_module_key(param_name: str) -> str:
        """same as above"""
        if param_name.endswith(".weight"):
            return param_name[:-len(".weight")]
        if param_name.endswith(".bias"):
            return param_name[:-len(".bias")]
        return param_name

    def get_lora_param_dict(model):
        """same as above"""
        out = {}
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            lname = name.lower()
            if ("lora_a" in lname or "lora_b" in lname) and (name.endswith(".weight") or name.endswith(".bias")):
                out[name] = p
        return out

    # loop over all LoRA parameters
    lora_params = get_lora_param_dict(policy)

    diag_vars_full = {}
    diag_vars_resid = {}
    rank_dirs = {}
    rank_lambdas = {}

    s2 = float(prior_var)
    inv_s2 = 1.0 / max(s2, eps)

    for pname, p in lora_params.items():
        device = p.device
        
        # Map parameter level Laplace to module level K-FAC
        module_key = param_to_module_key(pname)

        if module_key not in factors:
            # fallback small variance (if no factors found)
            diag_vars_full[pname]  = torch.full_like(p, 1e-3)
            diag_vars_resid[pname] = torch.full_like(p, 1e-3)
            rank_dirs[pname] = []
            rank_lambdas[pname] = []
            continue

        A, S = factors[module_key]

        # Move to device, float32 for stability
        A = A.to(device=device, dtype=torch.float32)
        S = S.to(device=device, dtype=torch.float32)

        # diag extraction (supports both full matrix and diag vector)
        if A.dim() == 2:
            A_diag = torch.diag(A)
        else:
            A_diag = A

        if S.dim() == 2:
            S_diag = torch.diag(S)
        else:
            S_diag = S

        A_diag = A_diag.clamp_min(eps)
        S_diag = S_diag.clamp_min(eps)

        # Fisher diagonal under Kronecker approx for weight matrix: diag(S) x diag(A) 
        F_diag = torch.outer(S_diag, A_diag).clamp_min(eps) # shape [d_out, d_in]

        # posterior diag variance: 1/(F + 1/s2)
        post_var_full = (1.0 / (F_diag + inv_s2)).clamp_min(eps)
        diag_vars_full[pname] = post_var_full.to(dtype=p.dtype)

        # rank-k directions (ONLY if we have full A and S)
        can_rank = (A.dim() == 2) and (S.dim() == 2) and (k is not None) and (k > 0)

        if not can_rank:
            # diag-only mode
            rank_dirs[pname] = []
            rank_lambdas[pname] = []
            diag_vars_resid[pname] = post_var_full.to(dtype=p.dtype)
            continue

        # eigenvectors of A and S for low-rank directions u = vS x vA (outer)
        evalsA, evecsA = stable_eigh_topk(A, k=min(rank_eigs_A, A.size(0)))
        evalsS, evecsS = stable_eigh_topk(S, k=min(rank_eigs_S, S.size(0)))

        # LoRA directions
        combos = []
        for i in range(evalsS.numel()):
            for j in range(evalsA.numel()):
                lam = float(evalsS[i].clamp_min(0).item() * evalsA[j].clamp_min(0).item())
                vS = evecsS[:, i]
                vA = evecsA[:, j]
                u = torch.ger(vS, vA) # [d_out, d_in]
                u = u / (u.norm() + eps)
                combos.append((lam, u))

        combos.sort(key=lambda x: x[0], reverse=True)
        combos = combos[:k]

        dirs = [u for (lam, u) in combos]
        # avoid redun
        dirs = orthonormalize_dirs_frobenius(dirs, eps_local=eps)

        lams = [float(lam) for (lam, _) in combos][:len(dirs)]

        rank_dirs[pname] = [u.to(dtype=p.dtype) for u in dirs]
        rank_lambdas[pname] = lams

        # residual diag subtraction
        if not use_resid_subtraction:
            diag_vars_resid[pname] = post_var_full.to(dtype=p.dtype)
        else:
            resid = post_var_full.clone()
            for u, lam in zip(rank_dirs[pname], rank_lambdas[pname]):
                # posterior variance along direction u under Laplace: Var(alpha) = 1/(lam + 1/s2)
                var_dir = 1.0 / (float(lam) + inv_s2)
                # subtract the variance mass projected onto each coordinate (u^2)
                resid = resid - float(var_dir) * (u.to(resid.dtype) ** 2)
            resid = resid.clamp_min(eps)
            diag_vars_resid[pname] = resid.to(dtype=p.dtype)

    return diag_vars_full, diag_vars_resid, rank_dirs, rank_lambdas



In [14]:
for n,p in policy.named_parameters():
    if "lora" in n.lower():
        print("LORA PARAM:", n, p.shape)


LORA PARAM: base_model.model.transformer.h.0.attn.c_attn.lora_A.default.weight torch.Size([8, 768])
LORA PARAM: base_model.model.transformer.h.0.attn.c_attn.lora_B.default.weight torch.Size([2304, 8])
LORA PARAM: base_model.model.transformer.h.0.attn.c_proj.lora_A.default.weight torch.Size([8, 768])
LORA PARAM: base_model.model.transformer.h.0.attn.c_proj.lora_B.default.weight torch.Size([768, 8])
LORA PARAM: base_model.model.transformer.h.0.mlp.c_proj.lora_A.default.weight torch.Size([8, 3072])
LORA PARAM: base_model.model.transformer.h.0.mlp.c_proj.lora_B.default.weight torch.Size([768, 8])
LORA PARAM: base_model.model.transformer.h.1.attn.c_attn.lora_A.default.weight torch.Size([8, 768])
LORA PARAM: base_model.model.transformer.h.1.attn.c_attn.lora_B.default.weight torch.Size([2304, 8])
LORA PARAM: base_model.model.transformer.h.1.attn.c_proj.lora_A.default.weight torch.Size([8, 768])
LORA PARAM: base_model.model.transformer.h.1.attn.c_proj.lora_B.default.weight torch.Size([768, 8])

In [ ]:
diag_full, diag_resid, rank_dirs, rank_lams = build_laplace_posterior_diag_and_rank4(
    policy=policy,
    factors=factors,
    prior_var=0.01, # s2
    k=k_rank)

print("state_dict keys:", len(policy.state_dict()))

sampled keys: 72
state_dict keys: 221


In [16]:
import json
import pandas as pd


files = ["batch17.json", "batch13.json", "batch12.json", "batch11.json", "batch0_cnndm.json",
        "batch3.json",
        "batch4.json",
        "batch5.json",
        "batch6.json",
        "batch7.json",
        "batch8.json",
        "batch9.json",
        "batch10.json",
        "batch11.json",
        "batch18.json",
        "batch19.json",
        "batch20.json",
        "batch22.json"]

data_pool = []
for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
        data_pool.extend(json.loads(line) for line in lines)


rows = []
for example in data_pool:
    post = example.get("info", {}).get("post", "")
    choice = example.get("choice", None)
    summaries = example.get("summaries", [])
    worker = example.get("worker", "")
    batch = example.get("batch", "")
    split = example.get("split", "")
    extra = example.get("extra", {})

    confidence = extra.get("confidence", None) if isinstance(extra, dict) else None

    if len(summaries) >= 2:
        summary_0 = summaries[0].get("text", "")
        summary_1 = summaries[1].get("text", "")
    else:
        summary_0 = ""
        summary_1 = ""

    rows.append({"post": post,
                "summary_0": summary_0,
                "summary_1": summary_1,
                "choice": choice,
                "worker": worker,
                "batch": batch,
                "split": split,
                "confidence": confidence})

df_pool = pd.DataFrame(rows)

In [17]:
cand = (df_pool[["worker", "post", "summary_0", "summary_1"]]
        .melt(id_vars=["worker", "post"],
            value_vars=["summary_0", "summary_1"],
            value_name="summary")
        .dropna(subset=["summary"])
        .drop_duplicates(["worker", "post", "summary"]))

# count distinct candidate per (worker, post)
cand_count = (cand.groupby(["worker", "post"])["summary"]
              .nunique()
              .reset_index(name="n_candidates"))

# filter only candidate >= 6
posts_ok = cand_count.query("n_candidates >= 6")[["worker", "post"]]

df_ok_posts = df_pool.merge(posts_ok, on=["worker", "post"], how="inner")

# count unique post per worker (after filter out candidate)
user_post_counts = (df_ok_posts[["worker", "post"]]
                    .drop_duplicates()
                    .groupby("worker")
                    .size()
                    .reset_index(name="n_unique_posts"))

# collect only user who have more than unique post >= 50
users_ok = user_post_counts.query("n_unique_posts >= 50")[["worker"]]

df_final = df_ok_posts.merge(users_ok, on="worker", how="inner")
df_final

,post,summary_0,summary_1,choice,worker,batch,split,confidence
0,On 3/31 I went to upgrade the outbound flight ...,United first class upgrade bait and switch by...,United is bait and switch.,0,RMwrIV50cNusBthNvLs1wSNdqFpQAg,batch17,valid2,2.0
1,On 3/31 I went to upgrade the outbound flight ...,United first class upgrade bait and switch by...,United baited me with a $50/seat 1st class up...,1,RMwrIV50cNusBthNvLs1wSNdqFpQAg,batch17,valid2,8.0
2,On 3/31 I went to upgrade the outbound flight ...,United baited me with a $50/seat 1st class up...,United is bait and switch.,0,RMwrIV50cNusBthNvLs1wSNdqFpQAg,batch17,valid2,2.0
3,On 3/31 I went to upgrade the outbound flight ...,United first class upgrade bait and switch by...,United is cheap and bait and switch me like t...,0,RMwrIV50cNusBthNvLs1wSNdqFpQAg,batch17,valid2,4.0
4,On 3/31 I went to upgrade the outbound flight ...,United is bait and switch.,United is cheap and bait and switch me like t...,1,RMwrIV50cNusBthNvLs1wSNdqFpQAg,batch17,valid2,8.0
...,...,...,...,...,...,...,...,...
4488,"To begin, and give context, it seems now two y...",I'm a college student who needs to move. I ne...,"I really like where I live, and want to move ...",0,KZL1qeRzHNYSfDAuOctL1iyVV8WC5N,batch22,valid2,4.0
4489,"To begin, and give context, it seems now two y...",I'm a college student who needs to move. I ne...,College student wants to move to his dream ci...,1,KZL1qeRzHNYSfDAuOctL1iyVV8WC5N,batch22,valid2,8.0
4490,"To begin, and give context, it seems now two y...",I'm a college student who needs to move. I ne...,A college student who really feels like he ne...,1,KZL1qeRzHNYSfDAuOctL1iyVV8WC5N,batch22,valid2,7.0
4491,"To begin, and give context, it seems now two y...","I really like where I live, and want to move ...",A college student who really feels like he ne...,1,KZL1qeRzHNYSfDAuOctL1iyVV8WC5N,batch22,valid2,7.0


In [31]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed_posts(df):
    posts = df["post"].unique().tolist()
    embs = embedder.encode(
        posts,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    return posts, dict(zip(posts, embs))

POSTS, POST_EMB = embed_posts(df)

from sklearn.cluster import AgglomerativeClustering

def cluster_posts_for_user(
    df,
    user,
    post_emb,
    distance_threshold=0.25  # ยิ่งเล็ก = ยิ่งเข้ม
):
    posts = df[df["worker"] == user]["post"].unique().tolist()
    if len(posts) < 2:
        return {0: posts}

    X = np.stack([post_emb[p] for p in posts])

    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="cosine",
        linkage="average",
        distance_threshold=distance_threshold
    )
    labels = clustering.fit_predict(X)

    clusters = {}
    for p, l in zip(posts, labels):
        clusters.setdefault(l, []).append(p)

    return clusters


for cid, posts in clusters.items():
    print(f"\n=== User {user_id} | Cluster {cid} | {len(posts)} posts ===")
    for p in posts:
        print("-", p[:120], "...")


Batches: 100%|██████████| 5/5 [00:03<00:00,  1.38it/s]


AttributeError: 'list' object has no attribute 'items'

In [19]:
workers_pass = users_ok["worker"].tolist()
len(workers_pass)

4

In [19]:
post_counts = (df_final[["worker", "post"]].drop_duplicates().groupby("worker")["post"].nunique())

keep_workers = post_counts[post_counts >= 50].index
df_200 = df_final[df_final["worker"].isin(keep_workers)].copy()


In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np

post_embedder = SentenceTransformer("all-MiniLM-L6-v2")

def build_post_embeddings(df):
    posts = df["post"].unique().tolist()
    embs = post_embedder.encode(
        posts,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    return dict(zip(posts, embs))

def split_posts_for_target(
    df,
    user,
    target_post,
    post_emb,
    n_val=3
):
    df_u = df[df["worker"] == user]
    posts = df_u["post"].unique().tolist()

    if target_post not in posts:
        raise ValueError("target_post not found for user")

    other_posts = [p for p in posts if p != target_post]
    if len(other_posts) == 0:
        return [], []

    tgt_emb = post_emb[target_post]

    sims = {
        p: float(tgt_emb @ post_emb[p])
        for p in other_posts
    }

    val_posts = sorted(sims, key=sims.get, reverse=True)[:n_val]
    warmup_posts = [p for p in other_posts if p not in val_posts]

    return warmup_posts, val_posts


def get_pairwise_rows(df, user, posts):
    return df[
        (df["worker"] == user) &
        (df["post"].isin(posts))
    ].copy()


def build_task_for_post(
    df,
    user,
    target_post,
    post_emb,
    n_val=3
):
    warmup_posts, val_posts = split_posts_for_target(
        df, user, target_post, post_emb, n_val
    )

    warmup_df = get_pairwise_rows(df, user, warmup_posts)
    interaction_df = get_pairwise_rows(df, user, [target_post])
    val_df = get_pairwise_rows(df, user, val_posts)

    return {
        "user": user,
        "target_post": target_post,
        "warmup_posts": warmup_posts,
        "val_posts": val_posts,
        "warmup_df": warmup_df,
        "interaction_df": interaction_df,
        "val_df": val_df
    }
POST_EMB = build_post_embeddings(df_200)


tasks = []

for user in df_200["worker"].unique():
    df_u = df_200[df_200["worker"] == user]
    posts = df_u["post"].unique().tolist()

    # กัน edge case
    if len(posts) < 3:
        continue

    for target_post in posts:
        task = build_task_for_post(
            df=df_200,
            user=user,
            target_post=target_post,
            post_emb=POST_EMB,
            n_val=3
        )

        # ต้องมี warm-up อย่างน้อย 1 post
        if len(task["warmup_df"]) == 0:
            continue

        tasks.append(task)


Batches: 100%|██████████| 11/11 [00:06<00:00,  1.78it/s]


In [22]:
task

{'user': 'KZL1qeRzHNYSfDAuOctL1iyVV8WC5N',
 'target_post': "So I have two sisters, one my age, and an older sister. Well lately I've been sleeping with my older sister and my other has been getting pissed at us. I get that it's fucked up but I got to fuck someone right? \n\nWell two weeks ago my older sister was staying over a friends so I was bored. I then went to give my sister her panties that somehow got in my laundry and when I opened her door she was naked on her bed sleep with the tv on... I don't know what the hell she was doing but her ass was just so plump... I couldn't even focus... I peaked out to see if my parents was sleep then slowly closed her door. That's when I started feeling it an oh my god... I thought she was flat at first because of our older sister but I guess she made up in ass. Anyway I looked over and saw her pussy and almost died... It was so juicy, like she felt me touching her. She wasn't wet, but it was just... Wow. Her breast were on the small side, but 

In [23]:
def count_candidates(interaction_df):
    return len(
        set(interaction_df["summary_0"])
        | set(interaction_df["summary_1"])
    )
n_candidates = count_candidates(task["interaction_df"])
print("candidates:", n_candidates)


candidates: 7


In [25]:
n_pairs = len(task["interaction_df"])
print("pairwise rows:", n_pairs)


pairwise rows: 9


In [28]:
def describe_task(task):
    df = task["interaction_df"]
    n_pairs = len(df)
    candidates = set(df["summary_0"]) | set(df["summary_1"])
    return {
        "user": task["user"],
        "post": task["target_post"],
        "n_candidates": len(candidates),
        "n_pairwise_available": n_pairs,
        "candidate_list": list(candidates)
    }

info = describe_task(task)
print(info)


{'user': 'KZL1qeRzHNYSfDAuOctL1iyVV8WC5N', 'post': "So I have two sisters, one my age, and an older sister. Well lately I've been sleeping with my older sister and my other has been getting pissed at us. I get that it's fucked up but I got to fuck someone right? \n\nWell two weeks ago my older sister was staying over a friends so I was bored. I then went to give my sister her panties that somehow got in my laundry and when I opened her door she was naked on her bed sleep with the tv on... I don't know what the hell she was doing but her ass was just so plump... I couldn't even focus... I peaked out to see if my parents was sleep then slowly closed her door. That's when I started feeling it an oh my god... I thought she was flat at first because of our older sister but I guess she made up in ass. Anyway I looked over and saw her pussy and almost died... It was so juicy, like she felt me touching her. She wasn't wet, but it was just... Wow. Her breast were on the small side, but still it

In [ ]:
unique_post_cnt = (df_train[["worker", "post"]].drop_duplicates().groupby("worker")["post"].nunique().reset_index(name="n_unique_posts"))

unique_post_cnt


,worker,n_unique_posts
0,KZL1qeRzHNYSfDAuOctL1iyVV8WC5N,82
1,OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca,49
2,RMwrIV50cNusBthNvLs1wSNdqFpQAg,93
3,gMlGeJl1vsMERrbmC7W717zpVevUh8,63


In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# collect only from train
df_train = df_split[df_split["split_y"] == "train"]

all_summaries = pd.concat([df_train["summary_0"],df_train["summary_1"],]).dropna().unique().tolist()

# embed + normalise
summary_embs = embedder.encode(all_summaries, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

# bank: summary -> embedding
SUMMARY_BANK = dict(zip(all_summaries, summary_embs))

def find_similar_summaries(base_summaries, summary_bank, k=3,):
    # collect embeddings that actually exist
    base_vecs = [
        summary_bank[s]
        for s in base_summaries
        if s in summary_bank
    ]

    if len(base_vecs) == 0:
        return []

    base_embs = np.stack(base_vecs)

    # compute similarity
    sims = {}
    for s, emb in summary_bank.items():
        sim = np.max(base_embs @ emb)
        sims[s] = sim

    # top-k excluding base
    extras = [s for s, _ in sorted(sims.items(), key=lambda x: x[1], reverse=True) if s not in base_summaries][:k]

    return extras


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
N_WARMUP_BUCKETS = 256
N_AL_BUCKETS = 4096
UNK_USER_TOKEN = "<UUNK>"

def hash_to_bucket(user_id: str, n_buckets: int) -> int:
    if user_id is None:
        return 0
    h = hashlib.md5(str(user_id).encode("utf-8")).hexdigest()
    return int(h, 16) % n_buckets

def normalize_user_token(raw_user_id: str, phase: str) -> str:
    if phase == "warmup":
        b = hash_to_bucket(raw_user_id, N_WARMUP_BUCKETS)
        return f"<W{b:03d}>"
    elif phase == "al":
        b = hash_to_bucket(raw_user_id, N_AL_BUCKETS)
        return f"<U{b:04d}>"
    return UNK_USER_TOKEN


def score_model_batch_chunked_dpo(policy, reference, tokenizer, post, candidates, device,
                                  *,
                                  user_id, phase="al", batch_size=16, params=None, beta=0.2, max_prompt_len=256, max_resp_len=256, max_length=512,mode="avg"):
    
    """cal DPO utility"""
    if params is None:
        params = {}
        
    user_tok = normalize_user_token(user_id, phase)

    posts = [post] * len(candidates)
    out = []
    for i in range(0, len(candidates), batch_size):
        out.extend(
            score_dpo_per_example_from_params(
                policy=policy,
                reference=reference,
                params=params,
                tokenizer=tokenizer,
                posts=posts[i:i+batch_size],
                summaries=candidates[i:i+batch_size],
                device=device,
                batch_size=batch_size,
                beta=beta,
                user_id=user_tok,
                phase=phase,
                max_prompt_len=max_prompt_len,
                max_resp_len=max_resp_len,
                max_length=max_length,
                mode=mode).tolist())
        
    return out

def build_test_pool_and_gt_by_worker(
    df_split,
    split_col="split_y",
    worker_col="worker",
    post_col="post",
    s0="summary_0",
    s1="summary_1",
    choice_col="choice",
    conf_col="confidence",
    n_expand=6):
    
    """Create a test pool and ground truth for each user"""
    test_df = df_split[df_split[split_col] != "train"].copy()

    keys = test_df[[worker_col, post_col]].drop_duplicates(keep="first").copy()
    keys["post_idx"] = keys.groupby(worker_col, sort=False).cumcount()

    def candidates_for_group(g):
        # base candidates (A,B,C,D)
        base = (g[[s0, s1]]
                .melt(value_name="summary")["summary"]
                .dropna()
                .unique()
                .tolist())

        # expand candidates (X,Y,Z)
        extra = find_similar_summaries(base_summaries=base, summary_bank=SUMMARY_BANK, k=n_expand)

        return base + extra

    def scores_for_group(g, candidates):
        score = {c: 0.0 for c in candidates}
        for r in g.itertuples(index=False):
            a = getattr(r, s0); b = getattr(r, s1)
            if pd.isna(a) or pd.isna(b):
                continue

            w = 1.0
            if conf_col in g.columns:
                cw = getattr(r, conf_col)
                if pd.notna(cw):
                    w = float(cw)

            if int(getattr(r, choice_col)) == 0:
                winner, loser = a, b
            else:
                winner, loser = b, a

            if winner in score: score[winner] += w
            if loser  in score: score[loser]  -= w

        return [score[c] for c in candidates]

    grouped = test_df.groupby([worker_col, post_col], sort=False)

    rows = []
    val_gt = {}

    for _, k in keys.iterrows():
        w = k[worker_col]; p = k[post_col]; idx = int(k["post_idx"])
        g = grouped.get_group((w, p))
        cands = candidates_for_group(g)
        scores = scores_for_group(g, cands)

        rows.append({worker_col: w,
                    "post_idx": idx,
                    post_col: p,
                    "candidates": cands})
        
        val_gt.setdefault(w, {})[idx] = scores

    pool_df = pd.DataFrame(rows).set_index([worker_col, "post_idx"]).sort_index()
    
    return pool_df, val_gt


def build_pair_label_lookup(df,
                            worker_col="worker", post_col="post",
                            s0="summary_0", s1="summary_1",
                            choice_col="choice", conf_col="confidence"):
    """pair_lookup from Train!!: (w,post,A,B)->(label,conf)"""
    
    df2 = df.copy()
    df2[conf_col] = pd.to_numeric(df2.get(conf_col, 0.0), errors="coerce").fillna(0.0)

    lookup = {}
    def put(key, label, conf):
        if key not in lookup or conf > lookup[key][1]:
            lookup[key] = (label, conf)

    for r in df2.itertuples(index=False):
        w = getattr(r, worker_col)
        p = getattr(r, post_col)
        a = getattr(r, s0)
        b = getattr(r, s1)
        c = int(getattr(r, choice_col))
        conf = float(getattr(r, conf_col))
        win = a if c == 0 else b

        put((w, p, a, b), 0 if win == a else 1, conf)
        put((w, p, b, a), 0 if win == b else 1, conf)

    return lookup

# GT transform for evaluate_ndcg_k (non-negative)
def gt_shift(scores):
    x = np.asarray(scores, dtype=float)
    return (x - x.min()).tolist()

def prepare_pools_and_gt(df_split):
    """prepare everything per-user (train pool, val df, gt scores, pair lookup)"""
    df_train = df_split[df_split["split_y"] == "train"].copy()
    df_val   = df_split[df_split["split_y"] != "train"].copy()

    pair_lookup = build_pair_label_lookup(df_train)

    tmp_train = df_train.copy()
    tmp_train["split_y"] = "test"
    train_pool_df, _ = build_test_pool_and_gt_by_worker(tmp_train, split_col="split_y")

    # val pool + val_gt
    val_pool_df, val_gt_by_user = build_test_pool_and_gt_by_worker(df_val, split_col="split_y")

    train_reset = train_pool_df.reset_index()[["worker", "post", "candidates", "post_idx"]]
    val_reset   = val_pool_df.reset_index()[["worker", "post", "candidates", "post_idx"]]

    USERS = sorted(val_reset["worker"].unique())

    train_pool_data_by_user = {}
    val_df_by_user = {}
    gt_scores_by_user = {}

    for u in USERS:
        train_pool_data_by_user[u] = (train_reset[train_reset["worker"] == u].sort_values("post_idx")[["post", "candidates"]].to_dict("records"))

        df_u = (val_reset[val_reset["worker"] == u].sort_values("post_idx")[["worker", "post_idx", "post", "candidates"]].reset_index(drop=True))
        val_df_by_user[u] = df_u

        gt_u = []
        for i in range(len(df_u)):
            pid = int(df_u.loc[i, "post_idx"])
            gt_u.append(gt_shift(val_gt_by_user[u][pid]))
        gt_scores_by_user[u] = gt_u

    return USERS, pair_lookup, train_pool_data_by_user, val_df_by_user, gt_scores_by_user, val_gt_by_user



USERS, pair_lookup, train_pool_data_by_user, val_df_by_user, gt_scores_by_user, val_gt_by_user = prepare_pools_and_gt(df_split)

12
 I slept with another guy after breaking up with my ex, but now want to get back with my ex. How to bring this up?


In [ ]:
def ndcg_at_k(pred_scores, gt_scores, k):
    """
    use gt_scores (non-negative)
    """
    pred_scores = np.asarray(pred_scores)
    gt_scores   = np.asarray(gt_scores)

    order = np.argsort(pred_scores)[::-1][:k]
    gains = gt_scores[order]

    discounts = 1.0 / np.log2(np.arange(2, 2 + len(gains)))
    dcg = np.sum(gains * discounts)

    ideal_order = np.argsort(gt_scores)[::-1][:k]
    ideal_gains = gt_scores[ideal_order]
    ideal_dcg = np.sum(ideal_gains * discounts)

    return float(dcg / ideal_dcg) if ideal_dcg > 0 else 0.0

@torch.no_grad()
def evaluate_ndcg_k(
    policy, reference, tokenizer, df, gt_scores, device,
    *,
    k=2, shuffle_within_row=True, shuffle_seed=10,
    mode="avg",
    beta=0.2,
    user_id=None,
    phase="al",
    batch_size=8, max_length=512, max_prompt_len=256, max_resp_len=256):
    """
    - Iterate over each (user, post) pair
    - Shuffle the candidates (for fairness)
    - Score them using the MAP policy
    - Compute NDCG@k

    Return the mean and the per-row results
    """

    policy.eval()
    reference.eval()

    rng = random.Random(shuffle_seed)
    ndcgs = []

    for i, row in df.iterrows():
        post = row["post"]
        cands = list(row["candidates"])
        gt = gt_scores[i]

        # important for fairness
        if shuffle_within_row:
            perm = list(range(len(cands)))
            rng.shuffle(perm)
            cands = [cands[j] for j in perm]
            gt    = [gt[j]    for j in perm]

        posts = [post] * len(cands)

        # score with DPO utility
        scores = score_dpo_per_example_from_params(
            policy=policy,
            reference=reference,
            params={}, # MAP policy
            tokenizer=tokenizer,
            posts=posts,
            summaries=cands,
            device=device,
            batch_size=batch_size,
            beta=beta,
            user_id=user_id,
            phase=phase,
            max_prompt_len=max_prompt_len,
            max_resp_len=max_resp_len,
            max_length=max_length,
            mode=mode)

        ndcgs.append(ndcg_at_k(scores, gt, k))

    return float(np.mean(ndcgs)), ndcgs, None



@torch.no_grad()
def evaluate_ndcg_multi_k(
    policy,
    reference,
    tokenizer,
    df,
    gt_scores,
    device,
    *,
    ks=(1, 2, 3),
    shuffle_within_row=True,
    shuffle_seed=10,
    mode="avg",
    beta=0.2,
    user_id=None,
    phase="al",
    batch_size=8,
    max_length=512,
    max_prompt_len=256,
    max_resp_len=256):
    """
    Evaluate multiple values of k in a single run
    """

    policy.eval()
    reference.eval()

    rng = random.Random(shuffle_seed)

    all_ndcgs = {f"ndcg@{k}": [] for k in ks}

    for i, row in df.iterrows():
        post = row["post"]
        cands = list(row["candidates"])
        gt = gt_scores[i]

        if shuffle_within_row:
            perm = list(range(len(cands)))
            rng.shuffle(perm)
            cands = [cands[j] for j in perm]
            gt    = [gt[j]    for j in perm]

        posts = [post] * len(cands)

        scores = score_dpo_per_example_from_params(
            policy=policy,
            reference=reference,
            params={},
            tokenizer=tokenizer,
            posts=posts,
            summaries=cands,
            device=device,
            batch_size=batch_size,
            beta=beta,
            user_id=user_id,
            phase=phase,
            max_prompt_len=max_prompt_len,
            max_resp_len=max_resp_len,
            max_length=max_length,
            mode=mode,
        )

        # compute multiple k
        for k in ks:
            all_ndcgs[f"ndcg@{k}"].append(
                ndcg_at_k(scores, gt, k)
            )

    mean_ndcgs = {k: float(np.mean(v)) for k, v in all_ndcgs.items()}

    return mean_ndcgs, all_ndcgs


u = USERS[0]
print(len(train_pool_data_by_user[u][0]["candidates"]))
print(train_pool_data_by_user[u][0]["candidates"][-1])


In [28]:
print(train_pool_data_by_user[u][0]["candidates"][1])


 boyfriend broke up with me, but we're still kind of together by kissing and cuddling/sexing. I love him, what should I do?


In [ ]:
model_name = "gpt2"
global_lora_dir = "gpt2-dpo-lora"

tokenizer = AutoTokenizer.from_pretrained(global_lora_dir)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

base = AutoModelForCausalLM.from_pretrained(model_name).to(device)

base.resize_token_embeddings(len(tokenizer))

policy_G = PeftModel.from_pretrained(base, global_lora_dir).to(device)
policy_G.config.use_cache = False
policy_G.eval()


reference_policy = deepcopy(reference)

# personal policy (train LoRA only)
personal_policy = deepcopy(policy_G)

for name, p in personal_policy.named_parameters():
    p.requires_grad = ("lora" in name.lower())  # train only LoRA
personal_policy.train()


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(54610, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
           

In [ ]:

def topk_global_dirs(rank_dirs, rank_lambdas, prior_var=0.01, K=8):
    """
    Aggregate rank-K Laplace directions from all LoRA parameters
    We then select only the directions with the largest posterior variance, as these correspond to the most important sources of uncertainty.
    Mathematically, for a direction u, the Laplace posterior variance along that direction is: Var(alpha_u) = 1 / (lambda_u + 1 / sigma^2)
    where:
    - lambda_u is the curvature (eigenvalue of the Hessian/Fisher) along u,
    - sigma^2 is the prior variance.

    We keep the top-K directions with the largest Var(alpha_u), since these directions contribute the most to posterior uncertainty.
    """
    inv_s2 = 1.0 / float(prior_var)
    all_dirs = []
    for pname, dirs in rank_dirs.items():
        lams = rank_lambdas.get(pname, [])
        for u, lam in zip(dirs, lams):
            var_dir = 1.0 / (float(lam) + inv_s2)
            all_dirs.append((var_dir, pname, u))
    all_dirs.sort(key=lambda x: x[0], reverse=True)
    return [(pname, u, float(var_dir)) for var_dir, pname, u in all_dirs[:K]]

def params_plus_minus_for_one_dir(policy, pname, u, eps):
    """Approximate the derivative of the utility along direction u in parameter space
    If we move theta slightly along direction u, how does the utility change?
    """
    p = dict(policy.named_parameters())[pname]
    base = p.data
    u = u.to(device=base.device, dtype=base.dtype)
    return (
        {pname: (base + eps * u).clone()},
        {pname: (base - eps * u).clone()},
    )

def sample_residual_direction(policy, diag_vars_resid):
    """Sample parameter perturbations from the Laplace posterior in the non rank-K part (the residual diagonal component)"""
    delta = {}
    for name, p in get_lora_param_dict(policy).items():
        if name not in diag_vars_resid:
            continue
        var = diag_vars_resid[name].to(p.device, dtype=torch.float32)
        std = torch.sqrt(var.clamp_min(1e-12))
        eps = torch.randn_like(p, dtype=torch.float32)
        delta[name] = (eps * std).to(p.dtype)
    return delta

def params_plus_minus_from_delta(policy, delta_dict, eps_step):
    """
    Construct a parameter dictionary for swapping: theta +/- eps_step * delta_theta.
    Here, delta_theta is sampled from the residual (non rank-K) diagonal part of the Laplace posterior. We convert this sampled perturbation into two parameter
    settings in order to compute a central-difference estimate of the utility.
    This allows us to approximate the effect of the remaining (residual) parameter uncertainty on the utility that is not captured by the rank-K directions.

    """
    plus, minus = {}, {}
    named = dict(policy.named_parameters())
    for name, d in delta_dict.items():
        base = named[name].data
        plus[name]  = (base + eps_step * d).clone()
        minus[name] = (base - eps_step * d).clone()
    return plus, minus

def expected_rms_from_diag(diag_vars_resid):
    """
    We estimate the expected RMS magnitude of residual parameter perturbations from the diagonal Laplace variances to set an appropriate perturbation scale
    """
    vals = [float(v.float().mean().item()) for v in diag_vars_resid.values()]
    return math.sqrt(max(np.mean(vals), 1e-12))


@torch.no_grad()
def compute_posterior_over_utilities_diag_plus_rankk(
    policy, reference, tokenizer, post, candidates, device,
    diag_vars_resid, rank_dirs, rank_lambdas,
    prior_var=0.01, beta=0.2,
    K=8,
    eps_scale=0.2, # init eps for rank directions (adaptive)
    M_resid=20, # MC probes for residual diag
    resid_eps_abs=1e-3, #fixed small eps for residual
    resid_eps_max=1e-2, # cap
    max_length=512, batch_size=8,
    mode="avg", user_id=None,
    max_prompt_len=256,
    max_resp_len=256,
    jitter=1e-6,

    # adaptive epsilon for rank-dirs only
    adaptive_rank=True,
    target_rel=0.01,
    target_rel_lo=0.003,
    target_rel_hi=0.03,
    target_abs_floor=1e-4,
    eps_min=1e-6,
    eps_max=5e-2,
    max_adjust_steps=8,
    use_median=False,
    verbose=False):
    """
    Construct the distribution of candidate-summary utilities under uncertainty in the LoRA parameters, as approximated by the Laplace posterior

    - Compute mean utility (MAP)
    - Rank-K analytic propagation (principal axes)
    - Residual Monte Carlo propagation (diagonal)
    - PSD stabilization
    """

    policy.eval()
    reference.eval()

    posts = [post] * len(candidates)

    # mean utility (MAP)
    mu = score_dpo_per_example_from_params(
        policy=policy, reference=reference, params={},
        tokenizer=tokenizer, posts=posts, summaries=candidates, device=device,
        max_length=max_length, batch_size=batch_size,
        mode=mode, beta=beta, user_id=user_id,
        max_prompt_len=max_prompt_len, max_resp_len=max_resp_len).astype(np.float64)

    N = len(mu)
    C = np.zeros((N, N), dtype=np.float64)

    # scale for rank-dir adaptive
    u_std = float(np.std(mu))
    u_abs = float(np.mean(np.abs(mu))) + 1e-12
    u_scale = max(u_std, 0.1 * u_abs, 1e-6)

    target_lo  = max(target_rel_lo * u_scale, 0.3 * target_abs_floor)
    target_hi  = max(target_rel_hi * u_scale, 3.0 * target_abs_floor)
    target_mid = max(target_rel    * u_scale,       target_abs_floor)

    named_params = dict(policy.named_parameters())

    def _eval_util(params_dict):
        return score_dpo_per_example_from_params(
            policy=policy, reference=reference, params=params_dict,
            tokenizer=tokenizer, posts=posts, summaries=candidates, device=device,
            max_length=max_length, batch_size=batch_size,
            mode=mode, beta=beta, user_id=user_id,
            max_prompt_len=max_prompt_len, max_resp_len=max_resp_len,
        ).astype(np.float64)

    def _du_stat(u_plus, u_minus):
        d = np.abs(u_plus - u_minus)
        return float(np.median(d)) if use_median else float(np.mean(d))

    def _update_eps_ratio(eps, du):
        ratio = target_mid / max(du, 1e-12)
        ratio = float(np.clip(ratio, 0.2, 5.0))
        return float(np.clip(eps * ratio, eps_min, eps_max))

    # rank-dir adaptive
    def _adaptive_central_diff_for_dir(pname, u_dir, eps_init):
        """
        The Fisher/Hessian indicates that uncertainty is not distributed uniformly across all directions in parameter space. Instead, there exist a few
        principal axes along which:
                                    - the curvature is low,
                                    - the posterior variance is high,
                                    - and the impact on predictions is significant.
        Therefore, we:
        1) compute rank-K eigen-directions (principal directions)
        2) select only the top-K axes with the largest posterior variance

        This selection becomes feasible when using the full K-FAC approximation which provides access to these global curvature directions
        """
        eps = float(np.clip(eps_init, eps_min, eps_max))
        u_plus = u_minus = None

        for _ in range(max_adjust_steps if adaptive_rank else 1):
            plus, minus = params_plus_minus_for_one_dir(policy, pname, u_dir, eps)
            u_plus  = _eval_util(plus)
            u_minus = _eval_util(minus)
            du = _du_stat(u_plus, u_minus)

            if not adaptive_rank:
                break

            if du < target_lo or du > target_hi:
                new_eps = _update_eps_ratio(eps, du)
                if new_eps == eps:
                    break
                eps = new_eps
                continue
            break

        z = (u_plus - u_minus) / (2.0 * eps)

        if verbose:
            du = _du_stat(u_plus, u_minus)
            print(f"[rank-dir] {pname} eps={eps:.3e} du={du:.3e} target~{target_mid:.3e}")

        return z

    # residual 1-step linear (NO adaptive)
    def _central_diff_for_delta_noadapt(delta_dict, eps):
        
        plus, minus = params_plus_minus_from_delta(policy, delta_dict, eps)
        u_plus  = _eval_util(plus) # map to utility space via score_dpo_per_example_from_params
        u_minus = _eval_util(minus)
        z = (u_plus - u_minus) / (2.0 * eps)
        return z

    # rank-K global directions
    global_dirs = topk_global_dirs(rank_dirs, rank_lambdas, prior_var=prior_var, K=K) # The principal axes of the posterior in theta-space
    
    for pname, u_dir, var_dir in global_dirs:
        if pname not in named_params:
            continue
        eps_init = eps_scale * math.sqrt(max(var_dir, 1e-12))
        # If we slightly perturb the LoRA parameters along these axes, how much does the utility of each candidate change?
        z = _adaptive_central_diff_for_dir(pname, u_dir, eps_init)
        # update covariance
        C += float(var_dir) * np.outer(z, z)

    # residual diagonal probes (1-step linear, fixed eps)
    if M_resid > 0:
        C_res = np.zeros_like(C)
        n_eff = 0

        eps_resid = float(np.clip(resid_eps_abs, 1e-6, resid_eps_max))
        # if verbose:
        #     print(f"[resid-B] eps_resid={eps_resid:.3e} (fixed) M_resid={M_resid}")

        for _ in range(M_resid): # MONTE CARLO LOOP
            # the rank-K subspace covers only a small portion of the parameter space
            # the remaining parameters still carry uncertainty, so we sample residual directions and use a central-difference Monte Carlo estimate to capture
            # the expected effect of the remaining uncertainty
            delta = sample_residual_direction(policy, diag_vars_resid) # sampling parameter
            
            if len(delta) == 0:
                break
            
            z = _central_diff_for_delta_noadapt(delta, eps_resid)
            C_res += np.outer(z, z)
            n_eff += 1

        if n_eff > 0:
            C += C_res / float(n_eff) # SAMPLE AVERAGE

    # PSD guard
    C = 0.5 * (C + C.T)
    eigmin = float(np.min(np.linalg.eigvalsh(C)))
    if eigmin < 0:
        C += (-eigmin + jitter) * np.eye(N)

    return mu, C

In [1]:
def ensure_psd(C, eps=1e-6):
    C = 0.5 * (C + C.T)
    w, V = np.linalg.eigh(C)
    w = np.clip(w, eps, None)
    Cpsd = V @ (w[:, None] * V.T)
    return 0.5 * (Cpsd + Cpsd.T)

def binary_entropy(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


# IMP 
def imp_exact(mu, C):
    mu = np.asarray(mu, dtype=np.float64)
    C  = np.asarray(C,  dtype=np.float64)

    N = len(mu)
    b = int(np.argmax(mu))
    EI = np.zeros(N, dtype=np.float64)

    for a in range(N):
        if a == b:
            EI[a] = -np.inf
            continue
        m = mu[a] - mu[b]
        v = C[a, a] + C[b, b] - 2.0 * C[a, b]
        v = max(v, 1e-12)
        sd = np.sqrt(v)

        z = np.clip(m / sd, -10, 10)
        EI[a] = sd * (z * norm.cdf(z) + norm.pdf(z))  # Eq.7

    return EI, b

def rank_all_pairs_imp(mu, C):
    EI, b = imp_exact(mu, C)
    return sorted([((a, b), float(EI[a])) for a in range(len(mu)) if a != b],
                  key=lambda x: x[1], reverse=True)


def probit_p_from_muC(mu, C, a, b, eps=1e-12):
    m = float(mu[a] - mu[b])
    v = float(C[a,a] + C[b,b] - 2.0*C[a,b])
    v = max(v, eps)
    # Eq.4: p = Phi( m / sqrt(1 + v) )
    p = norm.cdf(m / np.sqrt(1.0 + v))
    return p, v



# UNPA (Eq.4-5)
def rank_all_pairs_unpa_probit(mu, C):
    mu = np.asarray(mu, dtype=np.float64)
    C  = np.asarray(C,  dtype=np.float64)
    N = len(mu)

    pairs = []
    for a in range(N):
        for b in range(a+1, N):
            p, _ = probit_p_from_muC(mu, C, a, b)
            score = 1.0 - abs(p - 0.5)
            pairs.append(((a,b), float(score)))
    return sorted(pairs, key=lambda x: x[1], reverse=True)


# EIG (Eq.6) WITHOUT Monte Carlo: deterministic Gauss-Hermite quadrature for E_f[ H(y|f) ]
def rank_all_pairs_eig_probit(mu, C, num_gh=20):
    mu = np.asarray(mu, dtype=np.float64)

    x, w = hermgauss(num_gh)
    w = w / np.sqrt(np.pi)

    N = len(mu)
    pairs = []

    for a in range(N):
        for b in range(a+1, N):
            m = float(mu[a] - mu[b])
            v = float(C[a,a] + C[b,b] - 2.0*C[a,b])
            v = max(v, 1e-12)

            # H(y|D): p_pred = Phi(m / sqrt(1+v))
            p_pred = norm.cdf(m / np.sqrt(1.0 + v))
            H = binary_entropy(p_pred)

            # E_f[H(y|f)]: d~N(m,v), p(y|d)=Phi(d)
            d = m + np.sqrt(v) * (np.sqrt(2.0) * x)
            p_cond = norm.cdf(d)
            Hc = float(np.sum(w * binary_entropy(p_cond)))

            pairs.append(((a,b), float(H - Hc)))

    return sorted(pairs, key=lambda x: x[1], reverse=True)




# TP (ใช้ sample แบบ deterministic sqrt(C) ได้ แต่ยังเป็น "sampling strategy" โดยนิยาม)
def rank_all_pairs_tp_probit(mu, C, num_gh=20, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)

    mu = np.asarray(mu, dtype=np.float64)
    N = len(mu)

    # Thompson sample: f ~ N(mu, C)
    w_eig, V = np.linalg.eigh(C)
    w_eig = np.clip(w_eig, 1e-12, None)
    L = V @ np.diag(np.sqrt(w_eig))
    f = mu + L @ rng.standard_normal(N)
    b = int(np.argmax(f))

    # GH for EIG
    x, w_gh = hermgauss(num_gh)
    w_gh = w_gh / np.sqrt(np.pi)


    pairs = []
    for a in range(N):
        if a == b:
            continue

        m = float(mu[a] - mu[b])
        v = float(C[a,a] + C[b,b] - 2.0*C[a,b])
        v = max(v, 1e-12)

        p_pred = norm.cdf(m / np.sqrt(1.0 + v))
        H = binary_entropy(p_pred)

        d = m + np.sqrt(v) * (np.sqrt(2.0) * x)
        p_cond = norm.cdf(d)
        Hc = float(np.sum(w_gh * binary_entropy(p_cond)))

        pairs.append(((a, b), float(H - Hc)))

    return sorted(pairs, key=lambda x: x[1], reverse=True)



# choose + controller
def choose_pair_from_ranklist(ranklist, used_pairs):
    for (a, b), score in ranklist:
        key = (min(a, b), max(a, b))
        if key not in used_pairs:
            used_pairs.add(key)
            return a, b, {"score": score}
    return None, None, {"error": "no pairs left"}


def acquisition_controller(
    strategy, post, candidates,
    policy, reference, tokenizer,
    diag_vars_resid, rank_dirs, rank_lambdas, device,
    used_pairs=None, rng=None,
    prior_var=0.01,
    # posterior settings
    mode="avg", K=K_dirs, M_resid=M_resid, eps_scale=0.2, eps_resid_scale=0.2,
    # EIG quadrature settings
    num_gh=20,
    # ✅ calibration
    tau=1.0, sigma2=1.0
):

    if rng is None:
        rng = np.random.default_rng(10)
    if used_pairs is None:
        used_pairs = set()

    N = len(candidates)

    if strategy == "random":
        for _ in range(200):
            a = int(rng.integers(0, N))
            b = int(rng.integers(0, N - 1))
            if b >= a:
                b += 1
            key = (min(a, b), max(a, b))
            if key not in used_pairs:
                used_pairs.add(key)
                return a, b, {"score": None}
        return None, None, {"error": "no pairs left"}

    mu, C = compute_posterior_over_utilities_diag_plus_rankk(
        policy=policy,
        reference=reference,
        tokenizer=tokenizer,
        post=post,
        candidates=candidates,
        device=device,
        diag_vars_resid=diag_vars_resid,
        rank_dirs=rank_dirs,
        rank_lambdas=rank_lambdas,
        adaptive_rank=True,
        verbose=True,
    )

    mu = np.asarray(mu, dtype=np.float64)
    C  = ensure_psd(np.asarray(C, dtype=np.float64), eps=1e-6)  # ✅ ทำ PSD guard ครั้งเดียว

    if strategy == "imp_lora_exact":
        ranklist = rank_all_pairs_imp(mu, C)
    elif strategy == "unpa_probit":
        ranklist = rank_all_pairs_unpa_probit(mu, C)
    elif strategy == "eig_probit":
        ranklist = rank_all_pairs_eig_probit(mu, C, num_gh=num_gh)
    elif strategy == "tp_probit":
        ranklist = rank_all_pairs_tp_probit(mu, C, num_gh=num_gh, rng=rng)
    else:
        ranklist = [((i, j), 0.0) for i in range(N) for j in range(i + 1, N)]
        rng.shuffle(ranklist)

    return choose_pair_from_ranklist(ranklist, used_pairs)


NameError: name 'K_dirs' is not defined

In [ ]:
@torch.no_grad()
def evaluate_pairwise_accuracy(
    policy,
    reference,
    tokenizer,
    df_pairs,          # DataFrame with columns: post, chosen_summary, rejected_summary
    device,
    *,
    beta=0.2,
    user_id=None,
    batch_size=8,
    max_length=512,
    max_prompt_len=256,
    max_resp_len=256,
):
    """
    Pairwise win-rate: fraction of pairs where model prefers chosen over rejected.
    This metric is DIRECTLY aligned with DPO.
    """
    policy.eval()
    reference.eval()

    wins = 0
    total = 0

    for row in df_pairs.itertuples():
        post = row.post
        y_pos = row.chosen_summary
        y_neg = row.rejected_summary

        u_pos = score_dpo_per_example_from_params(
            policy=policy,
            reference=reference,
            params={},
            tokenizer=tokenizer,
            posts=[post],
            summaries=[y_pos],
            device=device,
            beta=beta,
            user_id=user_id,
            max_length=max_length,
            max_prompt_len=max_prompt_len,
            max_resp_len=max_resp_len,
        )[0]

        u_neg = score_dpo_per_example_from_params(
            policy=policy,
            reference=reference,
            params={},
            tokenizer=tokenizer,
            posts=[post],
            summaries=[y_neg],
            device=device,
            beta=beta,
            user_id=user_id,
            max_length=max_length,
            max_prompt_len=max_prompt_len,
            max_resp_len=max_resp_len,
        )[0]

        wins += int(u_pos > u_neg)
        total += 1

    return wins / max(total, 1)

@torch.no_grad()
def evaluate_expected_utility(
    policy,
    reference,
    tokenizer,
    df,
    device,
    *,
    beta=0.2,
    user_id=None,
    batch_size=8,
    max_length=512,
    max_prompt_len=256,
    max_resp_len=256,
):
    """
    Mean DPO utility over all candidates.
    Measures how aligned the policy is with the reference-adjusted preference.
    """
    policy.eval()
    reference.eval()

    utilities = []

    for row in df.itertuples():
        post = row.post
        cands = list(row.candidates)
        posts = [post] * len(cands)

        u = score_dpo_per_example_from_params(
            policy=policy,
            reference=reference,
            params={},
            tokenizer=tokenizer,
            posts=posts,
            summaries=cands,
            device=device,
            beta=beta,
            user_id=user_id,
            max_length=max_length,
            max_prompt_len=max_prompt_len,
            max_resp_len=max_resp_len,
        )

        utilities.extend(list(u))

    return float(np.mean(utilities))

@torch.no_grad()
def evaluate_top1_success(
    policy,
    reference,
    tokenizer,
    df,
    gt_scores,
    device,
    *,
    beta=0.2,
    user_id=None,
    batch_size=8,
    max_length=512,
    max_prompt_len=256,
    max_resp_len=256,
):
    """
    Success@1:
    Does the model pick the best candidate (according to GT) as top-1?
    """
    policy.eval()
    reference.eval()

    success = 0
    total = 0

    for i, row in df.iterrows():
        post = row["post"]
        cands = list(row["candidates"])
        gt = np.asarray(gt_scores[i])

        posts = [post] * len(cands)

        scores = score_dpo_per_example_from_params(
            policy=policy,
            reference=reference,
            params={},
            tokenizer=tokenizer,
            posts=posts,
            summaries=cands,
            device=device,
            beta=beta,
            user_id=user_id,
            max_length=max_length,
            max_prompt_len=max_prompt_len,
            max_resp_len=max_resp_len,
        )

        pred_top = int(np.argmax(scores))
        gt_top   = int(np.argmax(gt))

        success += int(pred_top == gt_top)
        total += 1

    return success / max(total, 1)




In [ ]:
STRATEGY = "tp_probit"

policy_by_user = {}
val_history_by_user = {}
used_pairs_by_user = {}

for u in USERS:
    # global policy
    policy_u = deepcopy(policy_G)

    # train only LoRA
    for name, p in policy_u.named_parameters():
        p.requires_grad = ("lora" in name.lower())

    policy_u.train()

    policy_by_user[u] = policy_u
    used_pairs_by_user[u] = {} # track asked pairs per post
    val_history_by_user[u] = {
        "pair_acc": [],
        "exp_util": [],
        "top1": [],
        "ndcg@1": [],
        "ndcg@2": [],
        "ndcg@3": [],
    }



for u in USERS:
    policy_u = policy_by_user[u]

    mean_ndcgs, all_ndcgs = evaluate_ndcg_multi_k(
        policy=policy_u,
        reference=reference_policy,
        tokenizer=tokenizer,
        df=val_df_by_user[u],
        gt_scores=gt_scores_by_user[u],
        device=device,
        ks=(1, 2, 3),
        user_id=u)

    
    val_history_by_user[u]["ndcg@1"].append(mean_ndcgs["ndcg@1"])
    val_history_by_user[u]["ndcg@2"].append(mean_ndcgs["ndcg@2"])
    val_history_by_user[u]["ndcg@3"].append(mean_ndcgs["ndcg@3"])


    print(f"[{u}] " + ", ".join(f"{k.upper()} = {v:.4f}" for k, v in mean_ndcgs.items()))


[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4852, NDCG@2 = 0.5287, NDCG@3 = 0.5928
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5160, NDCG@2 = 0.6174, NDCG@3 = 0.6411
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4501, NDCG@2 = 0.4828, NDCG@3 = 0.5655
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4348, NDCG@2 = 0.4751, NDCG@3 = 0.5057


In [ ]:
# print("users:", USERS[:3])
# print("one row df:", val_df_by_user[USERS[0]].iloc[0])
# print("gt:", gt_scores_by_user[USERS[0]][0])

In [ ]:
policy_G.train()

for name, p in policy_G.named_parameters():
    if "lora" in name.lower():
        p.requires_grad_(True)

if STRATEGY != "random":
    policy_G.train()

    global_factors = calculate_kfac_factors_dpo(
        policy=policy_G,
        reference=reference_policy,
        loader=dataloader_train, # loader from global DPO
        device=device,
        tokenizer=tokenizer,
        beta=0.2)

    _, diag_resid_global, rank_dirs_global, rank_lams_global = build_laplace_posterior_diag_and_rank4(policy=policy_G, factors=global_factors, prior_var=0.01, k=k_rank)


In [63]:
NUM_ROUNDS = 5
SEEDS = [10, 1, 2, 3, 4]

def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def choose_unused_random_pair(
    N: int,
    used_pairs: set,
    rng: np.random.Generator,
    max_tries: int = 200,
):
    """
    Sample a candidate pair (A, B) uniformly at random and ensuring it has not been queried before
    """
    if N < 2:
        return None, None, {"error": "N < 2"}

    for _ in range(max_tries):
        a = int(rng.integers(0, N))
        b = int(rng.integers(0, N - 1))
        if b >= a:
            b += 1

        key = (min(a, b), max(a, b))
        if key not in used_pairs:
            used_pairs.add(key)
            return a, b, {"strategy": "random"}

    return None, None, {"error": "no unused pairs left"}
    
def run_one_seed(EXP_SEED):

    set_global_seed(EXP_SEED)

    # cold start per seed
    # personalised preference learning
    df_train_by_user = {u: [] for u in USERS}
    used_pairs_by_user = {u: {} for u in USERS}

    policy_by_user = {}
    for u in USERS:
        # each user, start from a globally pretrained policy and train only the LoRA parameters
        policy_u = deepcopy(policy_G)
        for name, p in policy_u.named_parameters():
            p.requires_grad = ("lora" in name.lower())
        policy_u.train()
        policy_by_user[u] = policy_u

    # If a Bayesian strategy is used: all users start from the same global Laplace prior posterior, as no personalised data is available initially.
    if STRATEGY != "random":
        diag_resid_by_user = {u: copy.deepcopy(diag_resid_global) for u in USERS}
        rank_dirs_by_user  = {u: copy.deepcopy(rank_dirs_global)  for u in USERS}
        rank_lams_by_user  = {u: copy.deepcopy(rank_lams_global)  for u in USERS}
    else:
        diag_resid_by_user = {}
        rank_dirs_by_user  = {}
        rank_lams_by_user  = {}
        
    val_history_by_user = {
        u: {
            "pair_acc": [],
            "exp_util": [],
            "top1": [],
            "ndcg@1": [],
            "ndcg@2": [],
            "ndcg@3": [],
        }
        for u in USERS
    }

    for u in tqdm(USERS, desc=f"Users(seed={EXP_SEED})"):

        for r in tqdm(range(NUM_ROUNDS), desc=f"Rounds (user={u[:6]}, seed={EXP_SEED})", leave=False):

            policy_u = policy_by_user[u]

            # query every post
            for post_id, data in enumerate(train_pool_data_by_user[u]):

                post = data["post"]
                cands = data["candidates"]

                if post_id not in used_pairs_by_user[u]:
                    used_pairs_by_user[u][post_id] = set()

                # deterministic rng per (seed, user, round, post)
                rng = np.random.default_rng(hash((EXP_SEED, u, r, post_id)) % (2**32))

                if STRATEGY == "random":
                    A, B, _ = choose_unused_random_pair(
                        N=len(cands),
                        used_pairs=used_pairs_by_user[u][post_id],
                        rng=rng)
                else:
                    A, B, _ = acquisition_controller(
                        strategy=STRATEGY,
                        post=post,
                        candidates=cands,
                        policy=policy_u,
                        reference=reference_policy,
                        tokenizer=tokenizer,
                        diag_vars_resid=diag_resid_by_user[u],
                        rank_dirs=rank_dirs_by_user[u],
                        rank_lambdas=rank_lams_by_user[u],
                        device=device,
                        used_pairs=used_pairs_by_user[u][post_id],
                        tau=2.0,
                        sigma2=1.0)

                if A is None:
                    continue

                yA, yB = cands[A], cands[B]
                
                # lookup winner and loser
                hit = pair_lookup.get((u, post, yA, yB))
                if hit is None:
                    continue

                label, conf = hit
                winner, loser = (yA, yB) if label == 0 else (yB, yA)
                
                # preference pair for dpo
                df_train_by_user[u].append({
                    "post": post,
                    "chosen_summary": winner,
                    "rejected_summary": loser,
                    "user_id": u,
                    "confidence": conf})
            train_df = pd.DataFrame(df_train_by_user[u])

            loader = DataLoader(
                GPT2PreferenceDataset(train_df, tokenizer, phase="al"),
                batch_size=4,
                shuffle=True,
                collate_fn=collate_fn,
                generator=torch.Generator().manual_seed(EXP_SEED * 100 + r))

            optimizer = AdamW((p for p in policy_u.parameters() if p.requires_grad), lr=1e-4)

            train_dpo_gpt2(policy=policy_u,
                            reference=reference_policy,
                            dataloader_train=loader,
                            dataloader_test=loader,
                            optimizer=optimizer,
                            device=device,
                            epochs=3) # get new MAP estimate

            if STRATEGY != "random":
                # update Laplace posterior
                new_factors = calculate_kfac_factors_dpo(
                    policy=policy_u,
                    reference=reference_policy,
                    loader=loader,
                    device=device,
                    tokenizer=tokenizer,
                    beta=0.2)

                _, diag_resid_u, rank_dirs_u, rank_lams_u = build_laplace_posterior_diag_and_rank4(
                    policy=policy_u,
                    factors=new_factors,
                    prior_var=0.01,
                    k=k_rank)

                diag_resid_by_user[u] = diag_resid_u
                rank_dirs_by_user[u]  = rank_dirs_u
                rank_lams_by_user[u]  = rank_lams_u
            
            # The Laplace approximation is not used during evaluation. It is applied only in the acquisition step (query selection)
            mean_ndcgs, _ = evaluate_ndcg_multi_k(
                policy=policy_by_user[u],
                reference=reference_policy,
                tokenizer=tokenizer,
                df=val_df_by_user[u],
                gt_scores=gt_scores_by_user[u],
                device=device, ks=(1, 2, 3), user_id=u)
            
            policy_u = policy_by_user[u]

            pair_acc = evaluate_pairwise_accuracy(
                policy_u, reference_policy, tokenizer,
                df_train_by_user[u], device,
                user_id=u
            )

            exp_util = evaluate_expected_utility(
                policy_u, reference_policy, tokenizer,
                val_df_by_user[u], device,
                user_id=u
            )

            top1 = evaluate_top1_success(
                policy_u, reference_policy, tokenizer,
                val_df_by_user[u], gt_scores_by_user[u], device,
                user_id=u
            )



            print(
                f"[{u}] "
                f"PairAcc={pair_acc:.3f} | "
                f"ExpUtil={exp_util:.3f} | "
                f"Top1={top1:.3f} | "
                f"NDCG@1={mean_ndcgs['ndcg@1']:.3f}"
            )

        
            val_history_by_user[u]["pair_acc"].append(pair_acc)
            val_history_by_user[u]["exp_util"].append(exp_util)
            val_history_by_user[u]["top1"].append(top1)
            val_history_by_user[u]["ndcg@1"].append(mean_ndcgs["ndcg@1"])
            val_history_by_user[u]["ndcg@2"].append(mean_ndcgs["ndcg@2"])
            val_history_by_user[u]["ndcg@3"].append(mean_ndcgs["ndcg@3"])

        
            print(
                f"Round {r} [{u}] "
                f"PairAcc={pair_acc:.3f} | "
                f"ExpUtil={exp_util:.3f} | "
                f"Top1={top1:.3f} | "
                f"NDCG@1={mean_ndcgs['ndcg@1']:.3f}"
            )

    return val_history_by_user
    
ALL_RESULTS = []

for seed in SEEDS:
    hist = run_one_seed(seed)
    ALL_RESULTS.append(hist)

Rounds (user=KZL1qe, seed=10):   0%|          | 0/5 [00:00<?, ?it/s]

Epochs:   0%|          | 0/3 [00:00<?, ?it/s]

Epochs:  33%|███▎      | 1/3 [00:00<00:00,  2.03it/s]

Epoch 1 | Loss: 0.8436 | Train Accuracy: 0.4545, Win Probability: 0.5453 | Validation Accuracy: 0.4545, Win Probability: 0.5453




Epochs:  67%|██████▋   | 2/3 [00:00<00:00,  2.04it/s]

Epoch 2 | Loss: 0.6173 | Train Accuracy: 0.5455, Win Probability: 0.5890 | Validation Accuracy: 0.5455, Win Probability: 0.5890




Epochs: 100%|██████████| 3/3 [00:01<00:00,  2.04it/s]


...

Epoch 1 | Loss: 0.6888 | Train Accuracy: 0.2500, Win Probability: 0.5100 | Validation Accuracy: 0.2500, Win Probability: 0.5100
Epoch 2 | Loss: 0.5684 | Train Accuracy: 0.2500, Win Probability: 0.5524 | Validation Accuracy: 0.2500, Win Probability: 0.5524




Epochs: 100%|██████████| 3/3 [00:00<00:00,  5.60it/s]


Epoch 3 | Loss: 0.5628 | Train Accuracy: 0.2500, Win Probability: 0.5943 | Validation Accuracy: 0.2500, Win Probability: 0.5943






                                                              
Rounds (user=OKFDIs, seed=10):  20%|██        | 1/5 [02:17<09:08, 137.15s/it]

[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5442, NDCG@2 = 0.5761, NDCG@3 = 0.6087


In [ ]:
# ts

[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4852, NDCG@2 = 0.5287, NDCG@3 = 0.5928
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5160, NDCG@2 = 0.6174, NDCG@3 = 0.6411
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4501, NDCG@2 = 0.4828, NDCG@3 = 0.5655
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4348, NDCG@2 = 0.4751, NDCG@3 = 0.5057


# eig_probit
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4852, NDCG@2 = 0.5287, NDCG@3 = 0.5928
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5035, NDCG@2 = 0.5217, NDCG@3 = 0.5827
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5412, NDCG@2 = 0.5049, NDCG@3 = 0.5800
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5235, NDCG@2 = 0.5536, NDCG@3 = 0.6074
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5377, NDCG@2 = 0.5813, NDCG@3 = 0.5909
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4065, NDCG@2 = 0.5450, NDCG@3 = 0.5789


[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5160, NDCG@2 = 0.6174, NDCG@3 = 0.6411
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5422, NDCG@2 = 0.5951, NDCG@3 = 0.6683
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5160, NDCG@2 = 0.5867, NDCG@3 = 0.6151
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5286, NDCG@2 = 0.5179, NDCG@3 = 0.6103
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.6097, NDCG@2 = 0.5990, NDCG@3 = 0.6107
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5753, NDCG@2 = 0.5927, NDCG@3 = 0.6190


[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4501, NDCG@2 = 0.4828, NDCG@3 = 0.5655
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5332, NDCG@2 = 0.5034, NDCG@3 = 0.5770
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5228, NDCG@2 = 0.5332, NDCG@3 = 0.6157
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4515, NDCG@2 = 0.4942, NDCG@3 = 0.5673
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4850, NDCG@2 = 0.5208, NDCG@3 = 0.5737
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4810, NDCG@2 = 0.5424, NDCG@3 = 0.5570



[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4348, NDCG@2 = 0.4751, NDCG@3 = 0.5057
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4715, NDCG@2 = 0.4772, NDCG@3 = 0.4934
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4414, NDCG@2 = 0.4634, NDCG@3 = 0.5087
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5031, NDCG@2 = 0.4832, NDCG@3 = 0.5153
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4461, NDCG@2 = 0.5073, NDCG@3 = 0.5020
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.3801, NDCG@2 = 0.4557, NDCG@3 = 0.5186




# unpa_probit

[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4852, NDCG@2 = 0.5287, NDCG@3 = 0.5928
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4805, NDCG@2 = 0.5691, NDCG@3 = 0.6061
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4734, NDCG@2 = 0.5331, NDCG@3 = 0.5392
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5791, NDCG@2 = 0.5517, NDCG@3 = 0.5710
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5258, NDCG@2 = 0.5342, NDCG@3 = 0.5446
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5246, NDCG@2 = 0.5332, NDCG@3 = 0.5221


[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5160, NDCG@2 = 0.6174, NDCG@3 = 0.6411
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5023, NDCG@2 = 0.5668, NDCG@3 = 0.6047
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.4279, NDCG@2 = 0.5424, NDCG@3 = 0.6141
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.4994, NDCG@2 = 0.5491, NDCG@3 = 0.6184
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.4994, NDCG@2 = 0.5187, NDCG@3 = 0.5876
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.4762, NDCG@2 = 0.5396, NDCG@3 = 0.5937


[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4501, NDCG@2 = 0.4828, NDCG@3 = 0.5655
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5183, NDCG@2 = 0.5336, NDCG@3 = 0.5981
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.6051, NDCG@2 = 0.6372, NDCG@3 = 0.6814
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.6121, NDCG@2 = 0.6267, NDCG@3 = 0.6581
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.6224, NDCG@2 = 0.6632, NDCG@3 = 0.6935
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.6271, NDCG@2 = 0.6802, NDCG@3 = 0.7044


[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4348, NDCG@2 = 0.4751, NDCG@3 = 0.5057
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4824, NDCG@2 = 0.5052, NDCG@3 = 0.5247
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5249, NDCG@2 = 0.5453, NDCG@3 = 0.5747
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5419, NDCG@2 = 0.5515, NDCG@3 = 0.5867
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5115, NDCG@2 = 0.5206, NDCG@3 = 0.5703
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4535, NDCG@2 = 0.5173, NDCG@3 = 0.5613



Expand val

In [ ]:
# expand val
# imp 
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4852, NDCG@2 = 0.5287, NDCG@3 = 0.5928
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4415, NDCG@2 = 0.5042, NDCG@3 = 0.5656
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4935, NDCG@2 = 0.5557, NDCG@3 = 0.5668
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4536, NDCG@2 = 0.5767, NDCG@3 = 0.5850
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5454, NDCG@2 = 0.6003, NDCG@3 = 0.6323
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5506, NDCG@2 = 0.5930, NDCG@3 = 0.6495


[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5160, NDCG@2 = 0.6174, NDCG@3 = 0.6411
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5083, NDCG@2 = 0.5763, NDCG@3 = 0.6251
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5083, NDCG@2 = 0.5346, NDCG@3 = 0.6008
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5071, NDCG@2 = 0.5842, NDCG@3 = 0.6058
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5583, NDCG@2 = 0.5996, NDCG@3 = 0.6573
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5490, NDCG@2 = 0.5857, NDCG@3 = 0.6554


[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4501, NDCG@2 = 0.4828, NDCG@3 = 0.5655
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.6087, NDCG@2 = 0.6226, NDCG@3 = 0.6777
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.6417, NDCG@2 = 0.6687, NDCG@3 = 0.7054
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5996, NDCG@2 = 0.6923, NDCG@3 = 0.7221
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5154, NDCG@2 = 0.6273, NDCG@3 = 0.6637
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.6456, NDCG@2 = 0.6519, NDCG@3 = 0.7218


[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4348, NDCG@2 = 0.4751, NDCG@3 = 0.5057
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4696, NDCG@2 = 0.4858, NDCG@3 = 0.5204
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4922, NDCG@2 = 0.4957, NDCG@3 = 0.5389
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5163, NDCG@2 = 0.5354, NDCG@3 = 0.5667
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5076, NDCG@2 = 0.5271, NDCG@3 = 0.5803
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5076, NDCG@2 = 0.5605, NDCG@3 = 0.5723


#STRATEGY = "random"
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4852, NDCG@2 = 0.5287, NDCG@3 = 0.5928
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4766, NDCG@2 = 0.4958, NDCG@3 = 0.5749
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5564, NDCG@2 = 0.5179, NDCG@3 = 0.5617
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.6029, NDCG@2 = 0.4983, NDCG@3 = 0.5475
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5988, NDCG@2 = 0.5049, NDCG@3 = 0.5524
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4939, NDCG@2 = 0.4853, NDCG@3 = 0.5614



[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5160, NDCG@2 = 0.6174, NDCG@3 = 0.6411
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5203, NDCG@2 = 0.5903, NDCG@3 = 0.6131
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5442, NDCG@2 = 0.5520, NDCG@3 = 0.6086
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.4776, NDCG@2 = 0.5113, NDCG@3 = 0.5659
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.4686, NDCG@2 = 0.5129, NDCG@3 = 0.6108
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.4466, NDCG@2 = 0.4996, NDCG@3 = 0.5949



[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4501, NDCG@2 = 0.4828, NDCG@3 = 0.5655
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.3896, NDCG@2 = 0.4790, NDCG@3 = 0.5693
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5824, NDCG@2 = 0.5531, NDCG@3 = 0.6318
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5414, NDCG@2 = 0.5767, NDCG@3 = 0.6268
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5414, NDCG@2 = 0.6282, NDCG@3 = 0.6441
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5540, NDCG@2 = 0.6245, NDCG@3 = 0.6410


[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4348, NDCG@2 = 0.4751, NDCG@3 = 0.5057
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4736, NDCG@2 = 0.4823, NDCG@3 = 0.5198
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4484, NDCG@2 = 0.4814, NDCG@3 = 0.5495
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5003, NDCG@2 = 0.5688, NDCG@3 = 0.5987
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5112, NDCG@2 = 0.5837, NDCG@3 = 0.6108



In [ ]:
# no expand (# คือ imp แต่ ไม่มี # คือ random)
# [KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4805, NDCG@2 = 0.5616, NDCG@3 = 0.5928
# R1 [KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5110, NDCG@2 = 0.5250, NDCG@3 = 0.5855
# R2 [KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5534, NDCG@2 = 0.5399, NDCG@3 = 0.5642
# R3 [KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4982, NDCG@2 = 0.5301, NDCG@3 = 0.5712
# R4 [KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4798, NDCG@2 = 0.5526, NDCG@3 = 0.5892
# R5 [KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.5269, NDCG@2 = 0.5758, NDCG@3 = 0.5944

[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4805, NDCG@2 = 0.5616, NDCG@3 = 0.5928
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4637, NDCG@2 = 0.5144, NDCG@3 = 0.5875
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.3618, NDCG@2 = 0.4469, NDCG@3 = 0.5305
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4174, NDCG@2 = 0.4556, NDCG@3 = 0.4898
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4302, NDCG@2 = 0.4400, NDCG@3 = 0.4984
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4221, NDCG@2 = 0.4541, NDCG@3 = 0.4730

# [OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5399, NDCG@2 = 0.6101, NDCG@3 = 0.6607
# [OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5142, NDCG@2 = 0.5779, NDCG@3 = 0.6216
# [OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.4932, NDCG@2 = 0.6100, NDCG@3 = 0.6419
# [OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.4954, NDCG@2 = 0.5645, NDCG@3 = 0.5850
# [OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5005, NDCG@2 = 0.6138, NDCG@3 = 0.6354
# [OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5005, NDCG@2 = 0.5210, NDCG@3 = 0.5757

[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5399, NDCG@2 = 0.6101, NDCG@3 = 0.6607
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5272, NDCG@2 = 0.6137, NDCG@3 = 0.6637
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5098, NDCG@2 = 0.5617, NDCG@3 = 0.5934
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.5460, NDCG@2 = 0.5808, NDCG@3 = 0.6237
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.4760, NDCG@2 = 0.5414, NDCG@3 = 0.5839
[OKFDIsAZl6Qa0m9x26f5Ao4S0uc7Ca] NDCG@1 = 0.4788, NDCG@2 = 0.5526, NDCG@3 = 0.5923




# [RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4501, NDCG@2 = 0.4828, NDCG@3 = 0.5655
# [RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4786, NDCG@2 = 0.5076, NDCG@3 = 0.6289
# [RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5667, NDCG@2 = 0.6736, NDCG@3 = 0.6759
# [RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.6025, NDCG@2 = 0.6759, NDCG@3 = 0.7028
# [RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5572, NDCG@2 = 0.6432, NDCG@3 = 0.6958
# [RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5186, NDCG@2 = 0.6280, NDCG@3 = 0.6615

[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4501, NDCG@2 = 0.4828, NDCG@3 = 0.5655
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5321, NDCG@2 = 0.4945, NDCG@3 = 0.5661
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5812, NDCG@2 = 0.4964, NDCG@3 = 0.5850
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4980, NDCG@2 = 0.5350, NDCG@3 = 0.6090
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.4584, NDCG@2 = 0.5871, NDCG@3 = 0.6214
[RMwrIV50cNusBthNvLs1wSNdqFpQAg] NDCG@1 = 0.5047, NDCG@2 = 0.5471, NDCG@3 = 0.6324


# [gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4348, NDCG@2 = 0.4751, NDCG@3 = 0.5030
# [gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5984, NDCG@2 = 0.5637, NDCG@3 = 0.5788
# [gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5485, NDCG@2 = 0.5995, NDCG@3 = 0.6402
# [gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.6465, NDCG@2 = 0.5917, NDCG@3 = 0.6534
# [gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4598, NDCG@2 = 0.5546, NDCG@3 = 0.6253
# [gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5094, NDCG@2 = 0.5782, NDCG@3 = 0.6555

[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4348, NDCG@2 = 0.4751, NDCG@3 = 0.5030
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.4852, NDCG@2 = 0.5378, NDCG@3 = 0.5558
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.6171, NDCG@2 = 0.6235, NDCG@3 = 0.6535
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5840, NDCG@2 = 0.6375, NDCG@3 = 0.6578
[gMlGeJl1vsMERrbmC7W717zpVevUh8] NDCG@1 = 0.5299, NDCG@2 = 0.6206, NDCG@3 = 0.6465
[KZL1qeRzHNYSfDAuOctL1iyVV8WC5N] NDCG@1 = 0.4866, NDCG@2 = 0.5455, NDCG@3 = 0.5985



# another baseline model

In [ ]:
from gppl.gp_classifier_svi import GPClassifierSVI

STRATEGY   = "gppl_imp"   # "bt_unc" | "gppl_imp"
NUM_ROUNDS = 5
SEEDS      = [10, 1, 2, 3, 4]

TAU, SIGMA2 = 1.0, 1.0

SBERT_MODEL_NAME = "all-MiniLM-L6-v2"
SBERT_MODEL = SentenceTransformer(SBERT_MODEL_NAME)

def ensure_psd(M, eps=1e-6):
    M = 0.5 * (M + M.T)
    eigvals = np.linalg.eigvalsh(M)
    if np.min(eigvals) < eps:
        M += np.eye(M.shape[0]) * (eps - np.min(eigvals))
    return M


def normalize_prior(scores):
    scores = np.asarray(scores, dtype=np.float64)
    return (scores - scores.mean()) / (scores.std() + 1e-8)


def build_sbert_features(post, candidates):
    texts = [f"Post:\n{post}\nSummary:\n{s}" for s in candidates]
    return SBERT_MODEL.encode(texts, normalize_embeddings=True)


def bt_fit_laplace_contextual(X, pair_labels, l2=1e-2, n_steps=50):
    N, D = X.shape

    if len(pair_labels) == 0:
        w  = np.zeros(D)
        Cw = np.eye(D) / l2
        mu = X @ w
        C  = X @ Cw @ X.T
        return mu, ensure_psd(C)

    w = np.zeros(D)
    wins  = np.asarray([a for a, b in pair_labels])
    loses = np.asarray([b for a, b in pair_labels])

    for _ in range(n_steps):
        Xa = X[wins]
        Xb = X[loses]
        d  = (Xa - Xb) @ w
        p  = expit(np.clip(d, -50, 50))
        q  = p * (1 - p)

        g = l2 * w + ((p - 1)[:, None] * (Xa - Xb)).sum(axis=0)

        H = l2 * np.eye(D)
        for i in range(len(q)):
            v = (Xa[i] - Xb[i])[:, None]
            H += q[i] * (v @ v.T)

        w -= np.linalg.solve(H, g)

    Cw = np.linalg.inv(H)
    mu = X @ w
    C  = X @ Cw @ X.T
    return mu, ensure_psd(C)


# GPPL UTIL
def build_gppl_training_data(features, pair_labels):
    X, y = [], []
    for a, b in pair_labels:
        X.append(features[a] - features[b]); y.append(1)
        X.append(features[b] - features[a]); y.append(0)
    if len(X) == 0:
        return None, None
    return np.asarray(X), np.asarray(y)


def get_gppl_model_cached(cache, user_id, post_id, features, pair_labels, prior_mu, max_iter):
    key = (user_id, post_id)

    if key not in cache:
        N, D = features.shape
        model = GPClassifierSVI(
            ninput_features=D,
            kernel_func="linear",
            ninducing=min(500, N),
            use_svi=True,
            verbose=False)
        model.mu0 = prior_mu.reshape(-1, 1)
        model._init_params(mu0=model.mu0)
        cache[key] = model

    model = cache[key]

    X, y = build_gppl_training_data(features, pair_labels)
    if X is not None and max_iter > 0:
        model.fit(X, y, max_iter=max_iter)

    f_hat, C = model._expec_f_output(
        Ks_starstar=np.ones(features.shape[0]),
        Ks_star=model.kernel_func(features, model.ls),
        mu0=model.mu0,
        full_cov=True)

    return f_hat.flatten(), C

# GPPL + IMP
def gppl_imp_acquisition(
    *,
    features, pair_labels, used_pairs, user_id, post_id, model_cache, prior_mu):
    mu, C = get_gppl_model_cached(cache=model_cache, user_id=user_id, post_id=post_id, features=features, pair_labels=pair_labels, prior_mu=prior_mu, max_iter=5)

    EI, b = imp_exact(mu, C)
    a = int(np.argmax(EI))

    if a == b:
        return None, None, None

    key = (min(a, b), max(a, b))
    if key in used_pairs:
        return None, None, None

    used_pairs.add(key)
    return b, a, float(EI[a])


def acquisition_controller_all(*, strategy, features, pair_labels, used_pairs, user_id, post_id, model_cache, prior_mu):
    if strategy == "bt_unc":
        mu, C = bt_fit_laplace_contextual(features, pair_labels)
        EI, b = imp_exact(mu, C)
        a = int(np.argmax(EI))
        return b, a, float(EI[a])

    if strategy == "gppl_imp":
        """
        For baseline evaluation, we use a zero-mean GP prior without updating from preference data. 
        During active learning, we initialise the GPPL model with a REAPER-based prior and update it using pairwise feedback
        """
        return gppl_imp_acquisition(features=features, pair_labels=pair_labels, used_pairs=used_pairs, user_id=user_id, post_id=post_id, model_cache=model_cache, prior_mu=prior_mu)

    raise ValueError(strategy)


def evaluate_ndcg_multi_k_baseline(strategy, user_id, df, gt_scores, pair_labels_by_post, sbert_cache, gppl_eval_cache,
                                   *, 
                                   ks=(1, 2, 3), shuffle_within_row=True, shuffle_seed=10):
    
    rng = random.Random(shuffle_seed)
    all_ndcgs = {f"ndcg@{k}": [] for k in ks}

    for i, row in df.iterrows():
        post = row["post"]
        cands = list(row["candidates"])
        gt = list(gt_scores[i])

        if shuffle_within_row:
            perm = list(range(len(cands)))
            rng.shuffle(perm)
            cands = [cands[j] for j in perm]
            gt    = [gt[j]    for j in perm]

        post_idx = int(row["post_idx"])
        pair_labels = pair_labels_by_post.get(post_idx, [])

        key = (user_id, post_idx)
        if key not in sbert_cache:
            sbert_cache[key] = build_sbert_features(post, cands)
        feats = sbert_cache[key]

        if strategy == "bt_unc":
            mu, _ = bt_fit_laplace_contextual(feats, pair_labels)

        elif strategy == "gppl_imp":
            prior_mu = np.zeros(len(cands), dtype=np.float64)
            mu, _ = get_gppl_model_cached(
                cache=gppl_eval_cache,
                user_id=user_id,
                post_id=post_idx,
                features=feats,
                pair_labels=pair_labels,
                prior_mu=prior_mu,
                max_iter=0,   # prior-only baseline)
        else:
            raise ValueError(strategy)

        mu = np.asarray(mu, dtype=float)
        gt = np.asarray(gt, dtype=float)

        for k in ks:
            all_ndcgs[f"ndcg@{k}"].append(ndcg_at_k(mu, gt, k))

    mean_ndcgs = {k: float(np.mean(v)) for k, v in all_ndcgs.items()}
    return mean_ndcgs, all_ndcgs


def run_one_seed(EXP_SEED):
    set_global_seed(EXP_SEED)

    gppl_model_cache = {}
    gppl_eval_cache  = {}

    pair_labels_by_user_post = {u: {} for u in USERS}
    used_pairs_by_user_post  = {u: {} for u in USERS}
    sbert_cache = {}

    val_history_by_user = {u: {"ndcg@1": [], "ndcg@2": [], "ndcg@3": []} for u in USERS}

    for u in tqdm(USERS, desc=f"Users(seed={EXP_SEED})"):

        mean_ndcgs, _ = evaluate_ndcg_multi_k_baseline(STRATEGY, u, val_df_by_user[u], gt_scores_by_user[u], pair_labels_by_user_post[u], sbert_cache, gppl_eval_cache)

        for k in mean_ndcgs:
            val_history_by_user[u][k].append(mean_ndcgs[k])

        for _ in tqdm(range(NUM_ROUNDS), leave=False):
            for post_id, data in enumerate(train_pool_data_by_user[u]):
                post = data["post"]
                cands = data["candidates"]

                pair_labels = pair_labels_by_user_post[u].setdefault(post_id, [])
                used_pairs  = used_pairs_by_user_post[u].setdefault(post_id, set())

                key = (u, post_id)
                if key not in sbert_cache:
                    sbert_cache[key] = build_sbert_features(post, cands)
                feats = sbert_cache[key]

                prior_mu = normalize_prior(reaper_score_lookup[(u, post_id)])

                A, B, _ = acquisition_controller_all(
                    strategy=STRATEGY,
                    features=feats,
                    pair_labels=pair_labels,
                    used_pairs=used_pairs,
                    user_id=u,
                    post_id=post_id,
                    model_cache=gppl_model_cache,
                    prior_mu=prior_mu)

                if A is None:
                    continue

                yA, yB = cands[A], cands[B]
                hit = pair_lookup.get((u, post, yA, yB))
                if hit is None:
                    hit = pair_lookup.get((u, post, yB, yA))
                    if hit is None:
                        continue
                    label = 1 - hit[0]
                else:
                    label = hit[0]

                pair_labels.append((A, B) if label == 0 else (B, A))

            mean_ndcgs, _ = evaluate_ndcg_multi_k_baseline(
                STRATEGY,
                u,
                val_df_by_user[u],
                gt_scores_by_user[u],
                pair_labels_by_user_post[u],
                sbert_cache,
                gppl_eval_cache)

            for k in mean_ndcgs:
                val_history_by_user[u][k].append(mean_ndcgs[k])

    return val_history_by_user


ALL_RESULTS = []
for seed in SEEDS:
    ALL_RESULTS.append(run_one_seed(seed))
